In [ ]:
# ============================================================
# BUSINESS STARTUP / SURVIVAL / FAILURE / JOB FLOWS DATA PROJECT
# Sources:
# 1. Census BDS  = startup/survival/failure proxy
# 2. BLS BED/BDM = openings/closings/job gains/job losses
# 3. U.S. Courts F-2 = bankruptcy business/nonbusiness filings
# 4. Census BFS = business applications/new business formation
# 5. YC = startup examples only
#
# Memory optimized:
# - streaming downloads
# - chunk processing for large BLS files
# - dtype optimization
# - raw + clean folder
# - progress, elapsed time, ETA
# ============================================================

# If missing packages, run this first in Jupyter:
# !pip install pandas requests beautifulsoup4 pdfplumber lxml tqdm pyarrow playwright
# !playwright install chromium

import os
import re
import gc
import json
import time
import math
import shutil
import zipfile
import warnings
from pathlib import Path
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path("business_startup_project")

RAW_DIR = BASE_DIR / "raw"
CLEAN_DIR = BASE_DIR / "clean"
LOG_DIR = BASE_DIR / "logs"

for p in [RAW_DIR, CLEAN_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Turn on/off sources
RUN_BDS = True
RUN_BLS_BDM = True
RUN_BANKRUPTCY = True
RUN_BFS = True
RUN_YC_EXAMPLES = False   # optional; set True only if you want YC example companies

# Safer default: only download useful BDS files, not every huge file
BDS_DOWNLOAD_ALL = False

# BLS BED/BDM filter
# "00" = U.S. totals
# "15" = Hawaii
# You can add more: ["00", "15", "06"]
BLS_STATE_CODES = ["00"]

# BLS: keep total private + 2-digit/sector groups
# "000000" = Total private
# You can leave None to keep all selected industries, but file gets bigger.
BLS_INDUSTRY_CODES = [
    "000000",  # Total private
    "100020",  # Construction
    "100030",  # Manufacturing
    "200020",  # Retail trade
    "200050",  # Information
    "200060",  # Financial activities
    "200070",  # Professional and business services
    "200080",  # Education and health services
    "200090",  # Leisure and hospitality
]

# BLS dataclass codes:
# 01 Gross Job Gains
# 03 Openings
# 04 Gross Job Losses
# 06 Closings
# 07 Establishment Births
# 08 Establishment Deaths
BLS_DATACLASS_CODES = ["01", "03", "04", "06", "07", "08"]

# Use AllItems for full history.
# Current is smaller but only current year-to-date.
BLS_DATA_FILE = "bd.data.1.AllItems"

# U.S. Courts F-2 bankruptcy years
BANKRUPTCY_MIN_YEAR = 1990
BANKRUPTCY_MAX_YEAR = 2026

# YC examples limit
YC_MAX_COMPANIES = 200

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; DataScienceStudentProject/1.0)"
}

session = requests.Session()
session.headers.update(HEADERS)


# ============================================================
# HELPERS: TIME / MEMORY / DOWNLOAD
# ============================================================

def now():
    return time.time()

def fmt_seconds(seconds):
    if seconds is None or math.isnan(seconds) or seconds < 0:
        return "unknown"
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h:
        return f"{h}h {m}m {s}s"
    if m:
        return f"{m}m {s}s"
    return f"{s}s"

def fmt_bytes(n):
    if n is None:
        return "unknown"
    n = float(n)
    for unit in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} TB"

def log_step(msg):
    print("\n" + "=" * 70)
    print(msg)
    print("=" * 70)

def download_file(url, out_path, overwrite=False, chunk_size=1024 * 1024):
    """
    Streaming download with progress, elapsed time, and ETA.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and not overwrite:
        print(f"Already exists, skip: {out_path} ({fmt_bytes(out_path.stat().st_size)})")
        return out_path

    tmp_path = out_path.with_suffix(out_path.suffix + ".part")

    start = now()
    with session.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0)) or None

        downloaded = 0
        last_print = 0

        with open(tmp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if not chunk:
                    continue
                f.write(chunk)
                downloaded += len(chunk)

                elapsed = now() - start
                speed = downloaded / elapsed if elapsed > 0 else 0

                if total:
                    pct = downloaded / total * 100
                    eta = (total - downloaded) / speed if speed > 0 else None
                    line = (
                        f"\rDownloading {out_path.name}: "
                        f"{pct:6.2f}% | {fmt_bytes(downloaded)}/{fmt_bytes(total)} | "
                        f"speed {fmt_bytes(speed)}/s | elapsed {fmt_seconds(elapsed)} | ETA {fmt_seconds(eta)}"
                    )
                else:
                    line = (
                        f"\rDownloading {out_path.name}: "
                        f"{fmt_bytes(downloaded)} | speed {fmt_bytes(speed)}/s | "
                        f"elapsed {fmt_seconds(elapsed)}"
                    )

                if now() - last_print > 0.5:
                    print(line, end="")
                    last_print = now()

    tmp_path.replace(out_path)
    print(f"\nSaved: {out_path} | total time: {fmt_seconds(now() - start)}")
    return out_path

def optimize_dataframe(df):
    """
    Basic memory optimization for pandas dataframe.
    """
    for col in df.columns:
        col_type = df[col].dtype

        if col_type == "object":
            # keep object if many unique strings, otherwise category saves memory
            nunique = df[col].nunique(dropna=False)
            total = len(df[col])
            if total > 0 and nunique / total < 0.5:
                df[col] = df[col].astype("category")

        elif pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")

        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")

    return df

def read_csv_safely(path, **kwargs):
    """
    Read CSV with safer defaults.
    """
    return pd.read_csv(
        path,
        low_memory=False,
        keep_default_na=False,
        **kwargs
    )

def save_clean(df, out_path):
    """
    Save clean file as CSV.
    Also tries Parquet if pyarrow is installed.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    df = optimize_dataframe(df)
    df.to_csv(out_path, index=False)

    try:
        parquet_path = out_path.with_suffix(".parquet")
        df.to_parquet(parquet_path, index=False)
        print(f"Saved CSV: {out_path}")
        print(f"Saved Parquet: {parquet_path}")
    except Exception as e:
        print(f"Saved CSV: {out_path}")
        print(f"Parquet skipped: {e}")

    return out_path


# ============================================================
# 1. CENSUS BDS
# Startup / survival / failure proxy
# ============================================================

def scrape_bds_links():
    """
    Scrape Census BDS page and find CSV dataset links.
    """
    url = "https://www.census.gov/data/datasets/time-series/econ/bds/bds-datasets.html"
    html = session.get(url, timeout=60).text
    soup = BeautifulSoup(html, "html.parser")

    rows = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        text = " ".join(a.get_text(" ", strip=True).split())

        if ".csv" in href.lower() and "bds" in href.lower():
            full_url = urljoin(url, href)
            rows.append({
                "source": "Census BDS",
                "name": text if text else Path(href).name,
                "url": full_url,
                "file_name": Path(full_url).name
            })

    df = pd.DataFrame(rows).drop_duplicates(subset=["url"])
    return df

def download_and_clean_bds():
    log_step("1. Census BDS: scrape links, download selected CSV files")

    links = scrape_bds_links()
    links_path = CLEAN_DIR / "bds_download_links.csv"
    links.to_csv(links_path, index=False)
    print(f"BDS links found: {len(links)}")
    print(f"Saved link list: {links_path}")

    if links.empty:
        print("No BDS CSV links found. Check Census page structure.")
        return

    # Useful files for startup / survival / failure proxy.
    # Download selected files by default to avoid too much data.
    keywords = [
        "Economy-wide",
        "State",
        "Sector",
        "State by Sector",
        "Firm Age",
        "Establishment Age",
        "State by Firm Age",
        "Sector by Firm Age",
        "State by Establishment Age",
        "Sector by Establishment Age",
    ]

    if BDS_DOWNLOAD_ALL:
        selected = links.copy()
    else:
        pattern = "|".join([re.escape(k) for k in keywords])
        selected = links[links["name"].str.contains(pattern, case=False, na=False)].copy()

        # keep unique URLs
        selected = selected.drop_duplicates(subset=["url"])

    print(f"BDS files selected: {len(selected)}")

    clean_index = []

    for i, row in selected.iterrows():
        url = row["url"]
        name = row["name"]
        file_name = row["file_name"]

        raw_path = RAW_DIR / "bds" / file_name
        clean_path = CLEAN_DIR / "bds" / file_name

        print(f"\nBDS file: {name}")
        download_file(url, raw_path, overwrite=False)

        try:
            # Most BDS files are manageable, but still read in chunks
            first = True
            total_rows = 0
            start = now()

            for chunk_i, chunk in enumerate(pd.read_csv(raw_path, chunksize=200_000, low_memory=False)):
                chunk["source_table"] = name
                chunk = optimize_dataframe(chunk)

                clean_path.parent.mkdir(parents=True, exist_ok=True)
                chunk.to_csv(
                    clean_path,
                    mode="w" if first else "a",
                    index=False,
                    header=first
                )

                total_rows += len(chunk)
                first = False

                elapsed = now() - start
                print(
                    f"\rCleaning {file_name}: chunk {chunk_i+1:,} | rows {total_rows:,} | elapsed {fmt_seconds(elapsed)}",
                    end=""
                )

            print(f"\nSaved clean BDS: {clean_path}")

            clean_index.append({
                "source": "Census BDS",
                "table_name": name,
                "raw_file": str(raw_path),
                "clean_file": str(clean_path),
                "rows": total_rows
            })

        except Exception as e:
            print(f"\nCould not clean {file_name}: {e}")

        gc.collect()

    pd.DataFrame(clean_index).to_csv(CLEAN_DIR / "bds_clean_file_index.csv", index=False)
    print(f"\nSaved BDS clean index: {CLEAN_DIR / 'bds_clean_file_index.csv'}")


# ============================================================
# 2. BLS BED / BDM
# Openings / closings / job gains / job losses
# ============================================================

def bls_url(file_name):
    return f"https://download.bls.gov/pub/time.series/bd/{file_name}"

def download_bls_bdm_files():
    log_step("2. BLS BED/BDM: download mapping files + big data file")

    files = [
        "bd.series",
        BLS_DATA_FILE,
        "bd.dataclass",
        "bd.dataelement",
        "bd.industry",
        "bd.ratelevel",
        "bd.state",
        "bd.sizeclass",
        "bd.periodicity",
        "bd.ownership",
        "bd.unitanalysis",
    ]

    paths = {}
    for file in files:
        url = bls_url(file)
        out_path = RAW_DIR / "bls_bdm" / file
        paths[file] = download_file(url, out_path, overwrite=False)

    return paths

def read_bls_mapping(path):
    """
    BLS mapping files are usually whitespace or tab separated.
    This tries tab first, then whitespace.
    """
    try:
        df = pd.read_csv(path, sep="\t", dtype=str, low_memory=False)
        if df.shape[1] == 1:
            df = pd.read_csv(path, sep=r"\s+", dtype=str, low_memory=False)
    except Exception:
        df = pd.read_csv(path, sep=r"\s+", dtype=str, low_memory=False)

    return df

def process_bls_bdm():
    paths = download_bls_bdm_files()

    log_step("2B. BLS BED/BDM: select series metadata")

    series_path = paths["bd.series"]

    # Read series metadata
    try:
        series = pd.read_csv(series_path, sep="\t", dtype=str, low_memory=False)
    except Exception:
        series = pd.read_csv(series_path, sep=r"\s+", dtype=str, low_memory=False)

    print(f"Series rows: {len(series):,}")
    print("Series columns:")
    print(series.columns.tolist())

    # Make sure columns exist
    needed_cols = [
        "series_id",
        "seasonal",
        "state_code",
        "industry_code",
        "dataelement_code",
        "sizeclass_code",
        "dataclass_code",
        "ratelevel_code",
        "periodicity_code",
        "ownership_code",
        "series_title",
    ]

    missing = [c for c in needed_cols if c not in series.columns]
    if missing:
        print("Missing columns in bd.series:", missing)
        print("Saving raw series preview for inspection.")
        series.head(50).to_csv(CLEAN_DIR / "bls_bdm_series_preview.csv", index=False)
        return

    # Filter: U.S./state, selected industries, job flow classes, level, quarterly, private
    selected = series.copy()

    selected = selected[selected["state_code"].isin(BLS_STATE_CODES)]
    selected = selected[selected["dataclass_code"].isin(BLS_DATACLASS_CODES)]
    selected = selected[selected["ratelevel_code"].eq("L")]
    selected = selected[selected["periodicity_code"].eq("Q")]
    selected = selected[selected["ownership_code"].eq("5")]
    selected = selected[selected["sizeclass_code"].eq("00")]

    if BLS_INDUSTRY_CODES is not None:
        selected = selected[selected["industry_code"].isin(BLS_INDUSTRY_CODES)]

    # Keep both employment and establishment counts if available
    selected_ids = set(selected["series_id"].astype(str))

    print(f"Selected BLS series: {len(selected):,}")

    selected_path = CLEAN_DIR / "bls_bdm_selected_series.csv"
    selected.to_csv(selected_path, index=False)
    print(f"Saved selected series: {selected_path}")

    if len(selected_ids) == 0:
        print("No BLS series selected. Relax filters.")
        return

    log_step("2C. BLS BED/BDM: process huge data file by chunks")

    data_path = paths[BLS_DATA_FILE]
    out_path = CLEAN_DIR / "bls_bdm_openings_closings_job_flows.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Mapping tables
    dataclass = read_bls_mapping(paths["bd.dataclass"])
    dataelement = read_bls_mapping(paths["bd.dataelement"])
    industry = read_bls_mapping(paths["bd.industry"])
    state = read_bls_mapping(paths["bd.state"])
    ratelevel = read_bls_mapping(paths["bd.ratelevel"])

    # Keep only useful columns if present
    meta = selected.copy()

    merge_maps = [
        (dataclass, "dataclass_code"),
        (dataelement, "dataelement_code"),
        (industry, "industry_code"),
        (state, "state_code"),
        (ratelevel, "ratelevel_code"),
    ]

    for map_df, key in merge_maps:
        if key in map_df.columns:
            meta = meta.merge(map_df, on=key, how="left")

    meta_cols = [
        "series_id",
        "seasonal",
        "state_code",
        "state_name",
        "industry_code",
        "industry_name",
        "dataelement_code",
        "dataelement_name",
        "dataclass_code",
        "dataclass_name",
        "ratelevel_code",
        "ratelevel_name",
        "series_title",
    ]

    meta_cols = [c for c in meta_cols if c in meta.columns]
    meta = meta[meta_cols].drop_duplicates("series_id")

    first = True
    total_read = 0
    total_kept = 0
    start = now()

    # BLS data format:
    # series_id year period value footnote_codes
    for chunk_i, chunk in enumerate(pd.read_csv(
        data_path,
        sep=r"\s+",
        dtype={
            "series_id": "string",
            "year": "int16",
            "period": "string",
            "value": "float32",
            "footnote_codes": "string",
        },
        chunksize=1_000_000,
        low_memory=False
    )):
        total_read += len(chunk)

        chunk = chunk[chunk["series_id"].isin(selected_ids)].copy()

        if len(chunk) > 0:
            chunk = chunk.merge(meta, on="series_id", how="left")
            chunk["quarter"] = chunk["period"].str.replace("Q", "", regex=False)
            chunk["quarter"] = pd.to_numeric(chunk["quarter"], errors="coerce").astype("Int8")

            chunk = optimize_dataframe(chunk)

            chunk.to_csv(
                out_path,
                mode="w" if first else "a",
                index=False,
                header=first
            )

            total_kept += len(chunk)
            first = False

        elapsed = now() - start
        rate = total_read / elapsed if elapsed > 0 else 0

        print(
            f"\rBLS chunks: {chunk_i+1:,} | read {total_read:,} rows | kept {total_kept:,} rows | "
            f"speed {rate:,.0f} rows/s | elapsed {fmt_seconds(elapsed)}",
            end=""
        )

        del chunk
        gc.collect()

    print(f"\nSaved clean BLS BED/BDM: {out_path}")
    print(f"Total rows read: {total_read:,}")
    print(f"Total rows kept: {total_kept:,}")

    # Summary table
    if out_path.exists():
        summary = pd.read_csv(out_path, low_memory=False)
        group_cols = [
            c for c in [
                "year",
                "quarter",
                "state_name",
                "industry_name",
                "dataclass_name",
                "dataelement_name"
            ]
            if c in summary.columns
        ]

        summary_out = (
            summary
            .groupby(group_cols, dropna=False)["value"]
            .sum()
            .reset_index()
            .sort_values(group_cols)
        )

        save_clean(summary_out, CLEAN_DIR / "bls_bdm_summary_by_year_quarter.csv")

        del summary, summary_out
        gc.collect()


# ============================================================
# 3. U.S. COURTS BANKRUPTCY TABLE F-2
# Business and nonbusiness cases filed
# ============================================================

def scrape_bankruptcy_f2_links():
    """
    Scrape F-2 links across available pagination.
    Saves PDF links even if PDF parsing fails.
    """
    base = "https://www.uscourts.gov/data-table-numbers/f-2"

    all_rows = []
    seen_urls = set()

    # The page has pagination. Try first 10 pages safely.
    for page_num in range(0, 10):
        url = base if page_num == 0 else f"{base}?page={page_num}"
        print(f"Scraping F-2 page: {url}")

        html = session.get(url, timeout=60).text
        soup = BeautifulSoup(html, "html.parser")

        page_rows_before = len(all_rows)

        for tr in soup.find_all("tr"):
            cells = [c.get_text(" ", strip=True) for c in tr.find_all(["td", "th"])]
            a = tr.find("a", href=True)

            if not a:
                continue

            href = a["href"]
            full_url = urljoin(url, href)

            if ".pdf" not in full_url.lower():
                continue

            row_text = " | ".join(cells)
            if "F-2" not in row_text and "Bankruptcy" not in row_text:
                continue

            # Find reporting period like March 31, 2026
            period_match = re.search(
                r"(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}",
                row_text
            )
            period = period_match.group(0) if period_match else ""

            year_match = re.search(r"\b(19|20)\d{2}\b", period)
            year = int(year_match.group(0)) if year_match else None

            if year and not (BANKRUPTCY_MIN_YEAR <= year <= BANKRUPTCY_MAX_YEAR):
                continue

            if full_url not in seen_urls:
                seen_urls.add(full_url)
                all_rows.append({
                    "source": "U.S. Courts F-2",
                    "reporting_period": period,
                    "year": year,
                    "url": full_url,
                    "file_name": Path(full_url).name
                })

        # Stop if page has no new PDF rows
        if len(all_rows) == page_rows_before and page_num > 0:
            break

        time.sleep(0.5)

    df = pd.DataFrame(all_rows).drop_duplicates(subset=["url"])
    if not df.empty:
        df = df.sort_values(["year", "reporting_period"], ascending=[False, False])

    return df

def parse_f2_pdf_total_row(pdf_path):
    """
    Extract only the national Total row from F-2 PDF.
    This avoids storing all district rows and keeps memory low.

    F-2 Total row usually has 14 numbers:
    total_all, total_ch7, total_ch11, total_ch13, total_other,
    business_all, business_ch7, business_ch11, business_ch13, business_other,
    nonbusiness_all, nonbusiness_ch7, nonbusiness_ch11, nonbusiness_ch13
    """
    try:
        import pdfplumber
    except Exception as e:
        raise ImportError("Install pdfplumber first: !pip install pdfplumber") from e

    text_all = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages[:2]:
            text = page.extract_text() or ""
            text_all += "\n" + text

    lines = text_all.splitlines()

    total_line = None
    for line in lines:
        clean = " ".join(line.split())
        if clean.startswith("Total "):
            total_line = clean
            break

    if not total_line:
        return None

    nums = re.findall(r"\d[\d,]*", total_line)
    nums = [int(x.replace(",", "")) for x in nums]

    if len(nums) < 14:
        return {
            "parse_status": "partial",
            "raw_total_line": total_line,
            "numbers_found": len(nums)
        }

    cols = [
        "total_all_chapters",
        "total_chapter_7",
        "total_chapter_11",
        "total_chapter_13",
        "total_other_chapters",
        "business_all_chapters",
        "business_chapter_7",
        "business_chapter_11",
        "business_chapter_13",
        "business_other_chapters",
        "nonbusiness_all_chapters",
        "nonbusiness_chapter_7",
        "nonbusiness_chapter_11",
        "nonbusiness_chapter_13",
    ]

    out = dict(zip(cols, nums[:14]))
    out["parse_status"] = "ok"
    out["raw_total_line"] = total_line
    return out

def download_and_clean_bankruptcy_f2():
    log_step("3. U.S. Courts Bankruptcy F-2: scrape PDF links, download, parse Total row")

    links = scrape_bankruptcy_f2_links()
    links_path = CLEAN_DIR / "bankruptcy_f2_pdf_links.csv"
    links.to_csv(links_path, index=False)

    print(f"F-2 PDF links found: {len(links):,}")
    print(f"Saved links: {links_path}")

    if links.empty:
        print("No F-2 links found.")
        return

    rows = []
    start_all = now()

    for idx, row in links.iterrows():
        url = row["url"]
        file_name = row["file_name"]
        raw_path = RAW_DIR / "bankruptcy_f2" / file_name

        print(f"\nBankruptcy F-2: {row['reporting_period']} | {file_name}")
        download_file(url, raw_path, overwrite=False)

        parsed = {
            "source": "U.S. Courts F-2",
            "reporting_period": row["reporting_period"],
            "year": row["year"],
            "url": url,
            "raw_file": str(raw_path),
        }

        try:
            total_data = parse_f2_pdf_total_row(raw_path)
            if total_data:
                parsed.update(total_data)
            else:
                parsed["parse_status"] = "no_total_row_found"
        except Exception as e:
            parsed["parse_status"] = "parse_error"
            parsed["error"] = str(e)

        rows.append(parsed)

        elapsed = now() - start_all
        print(f"Finished {len(rows):,}/{len(links):,} | elapsed {fmt_seconds(elapsed)}")

        gc.collect()

    df = pd.DataFrame(rows)
    save_clean(df, CLEAN_DIR / "bankruptcy_f2_total_row_clean.csv")


# ============================================================
# 4. CENSUS BFS
# Business applications / new business formation
# ============================================================

def download_and_clean_bfs():
    log_step("4. Census BFS: download monthly time series CSV and reshape to long format")

    bfs_url = "https://www.census.gov/econ/bfs/csv/bfs_monthly.csv"
    month_date_url = "https://www.census.gov/econ/bfs/csv/month_date_table.csv"

    raw_bfs = RAW_DIR / "bfs" / "bfs_monthly.csv"
    raw_dates = RAW_DIR / "bfs" / "month_date_table.csv"

    download_file(bfs_url, raw_bfs, overwrite=False)
    download_file(month_date_url, raw_dates, overwrite=False)

    print("\nReading BFS...")
    df = read_csv_safely(raw_bfs, dtype=str)
    print(f"BFS raw shape: {df.shape}")
    print("BFS columns:")
    print(df.columns.tolist())

    month_cols = [
        "jan", "feb", "mar", "apr", "may", "jun",
        "jul", "aug", "sep", "oct", "nov", "dec"
    ]

    existing_month_cols = [c for c in month_cols if c in df.columns]
    id_cols = [c for c in df.columns if c not in existing_month_cols]

    long_df = df.melt(
        id_vars=id_cols,
        value_vars=existing_month_cols,
        var_name="month_name",
        value_name="value_raw"
    )

    month_num_map = {
        "jan": 1, "feb": 2, "mar": 3, "apr": 4,
        "may": 5, "jun": 6, "jul": 7, "aug": 8,
        "sep": 9, "oct": 10, "nov": 11, "dec": 12
    }

    long_df["month"] = long_df["month_name"].map(month_num_map).astype("Int8")
    long_df["year"] = pd.to_numeric(long_df["year"], errors="coerce").astype("Int16")

    # Convert normal numbers; keep D/S/NA in value_raw
    long_df["value"] = (
        long_df["value_raw"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace({"NA": None, "D": None, "S": None, "": None})
    )
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")

    # Add date if month date table works
    try:
        dates = read_csv_safely(raw_dates, dtype=str)
        dates.columns = [c.strip().lower().replace(" ", "_") for c in dates.columns]

        # Expected columns could be year, month, start_date, end_date
        if "year" in dates.columns and "month" in dates.columns:
            dates["year"] = pd.to_numeric(dates["year"], errors="coerce").astype("Int16")
            dates["month"] = pd.to_numeric(dates["month"], errors="coerce").astype("Int8")
            long_df = long_df.merge(dates, on=["year", "month"], how="left")
    except Exception as e:
        print(f"Month date table merge skipped: {e}")

    # Save full long file
    save_clean(long_df, CLEAN_DIR / "bfs_monthly_long_clean.csv")

    # Save easy summary: U.S. total, seasonally adjusted, total NAICS
    easy = long_df.copy()

    if "geo" in easy.columns:
        easy = easy[easy["geo"].eq("US")]
    if "sa" in easy.columns:
        easy = easy[easy["sa"].eq("A")]
    if "naics_sector" in easy.columns:
        easy = easy[easy["naics_sector"].eq("TOTAL")]

    easy = easy.sort_values(["year", "month", "series"])
    save_clean(easy, CLEAN_DIR / "bfs_us_total_seasonally_adjusted.csv")

    del df, long_df, easy
    gc.collect()


# ============================================================
# 5. YC STARTUP EXAMPLES ONLY
# Optional Playwright scrape
# ============================================================

def scrape_yc_examples_playwright(max_companies=200):
    """
    Optional. YC directory is dynamic JS, so requests may not show company cards.
    This uses Playwright browser automation.

    This is examples only, not official survival/failure data.
    """
    try:
        from playwright.sync_api import sync_playwright
    except Exception as e:
        print("Playwright not installed.")
        print("Run:")
        print("!pip install playwright")
        print("!playwright install chromium")
        return pd.DataFrame()

    url = "https://www.ycombinator.com/companies?regions=United%20States%20of%20America"

    rows = []
    seen = set()

    log_step("5. YC examples only: browser scrape first company cards")

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page(user_agent=HEADERS["User-Agent"])
        page.goto(url, wait_until="networkidle", timeout=120_000)

        # Scroll to load cards
        for i in range(20):
            page.mouse.wheel(0, 3000)
            time.sleep(1)

            links = page.locator('a[href^="/companies/"]').all()
            print(f"\rScroll {i+1}/20 | company links visible: {len(links)}", end="")

            if len(links) >= max_companies:
                break

        print()

        links = page.locator('a[href^="/companies/"]').all()

        for link in links:
            try:
                href = link.get_attribute("href")
                text = link.inner_text(timeout=3000)

                if not href or href in seen:
                    continue

                seen.add(href)
                parts = [x.strip() for x in text.splitlines() if x.strip()]

                rows.append({
                    "source": "Y Combinator Directory",
                    "company_page": urljoin("https://www.ycombinator.com", href),
                    "raw_card_text": " | ".join(parts),
                })

                if len(rows) >= max_companies:
                    break
            except Exception:
                continue

        browser.close()

    df = pd.DataFrame(rows)
    out = CLEAN_DIR / "yc_startup_examples_only.csv"
    save_clean(df, out)

    print(f"YC examples saved: {out}")
    return df


# ============================================================
# FINAL RUNNER
# ============================================================

def run_all():
    project_start = now()

    print("PROJECT FOLDER:", BASE_DIR.resolve())
    print("Raw folder:", RAW_DIR.resolve())
    print("Clean folder:", CLEAN_DIR.resolve())

    if RUN_BDS:
        try:
            download_and_clean_bds()
        except Exception as e:
            print(f"BDS failed: {e}")

    if RUN_BLS_BDM:
        try:
            process_bls_bdm()
        except Exception as e:
            print(f"BLS BED/BDM failed: {e}")

    if RUN_BANKRUPTCY:
        try:
            download_and_clean_bankruptcy_f2()
        except Exception as e:
            print(f"Bankruptcy F-2 failed: {e}")

    if RUN_BFS:
        try:
            download_and_clean_bfs()
        except Exception as e:
            print(f"BFS failed: {e}")

    if RUN_YC_EXAMPLES:
        try:
            scrape_yc_examples_playwright(max_companies=YC_MAX_COMPANIES)
        except Exception as e:
            print(f"YC examples failed: {e}")

    elapsed = now() - project_start

    log_step("DONE")
    print(f"Total project time: {fmt_seconds(elapsed)}")
    print(f"Clean outputs are in: {CLEAN_DIR.resolve()}")

    # Print final clean files
    print("\nClean files created:")
    for f in sorted(CLEAN_DIR.rglob("*")):
        if f.is_file():
            print(f"- {f} | {fmt_bytes(f.stat().st_size)}")


# ============================================================
# RUN THIS
# ============================================================

run_all()

# One file

In [ ]:
# ============================================================
# ONEFILE BUSINESS STARTUP / SURVIVAL / JOB FLOWS / BANKRUPTCY PROJECT
#
# Everything saves inside one folder: OneFile
#
# Sources:
# 1. Census BDS  = startup / survival / failure proxy
# 2. BLS BED/BDM = openings / closings / job gains / job losses
# 3. U.S. Courts F-2 = bankruptcy business / nonbusiness filings
# 4. Census BFS = business applications / new business formation
# 5. YC examples = optional, examples only, not official survival data
#
# Memory optimized:
# - streaming downloads
# - chunk processing for large BLS file
# - no giant full-file load for BLS data
# - prints manual URLs if a website blocks Python
# ============================================================

# If packages are missing, run once in Jupyter:
# !pip install pandas requests beautifulsoup4 openpyxl pyarrow

import re
import gc
import json
import math
import time
import warnings
from pathlib import Path
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path("OneFile")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Turn sources on/off
RUN_BDS = True
RUN_BFS = True
RUN_BLS_BDM = True
RUN_BANKRUPTCY = True
RUN_YC_EXAMPLES = False  # optional; set True only if you want YC company examples

# BDS: use core files only, not hundreds of huge combinations
# These give economy-wide, state, industry/sector, and firm/establishment age views.
BDS_CORE_FILES = [
    "bds2023.csv",          # economy-wide, all years
    "bds2023_st.csv",       # state
    "bds2023_sec.csv",      # sector
    "bds2023_st_sec.csv",   # state by sector
    "bds2023_fa.csv",       # firm age
    "bds2023_st_fa.csv",    # state by firm age
    "bds2023_sec_fa.csv",   # sector by firm age
    "bds2023_ea.csv",       # establishment age
    "bds2023_st_ea.csv",    # state by establishment age
    "bds2023_sec_ea.csv",   # sector by establishment age
]

# BLS BED/BDM filter
# "00" = U.S. total, "15" = Hawaii. Add more if needed: ["00", "15", "06"]
BLS_STATE_CODES = ["00"]

# BLS industries: total private + major sectors
BLS_INDUSTRY_CODES = [
    "000000",  # Total private
    "100000",  # Goods-producing
    "100010",  # Natural resources and mining
    "100020",  # Construction
    "100030",  # Manufacturing
    "200000",  # Service-providing
    "200010",  # Wholesale trade
    "200020",  # Retail trade
    "200030",  # Transportation and warehousing
    "200040",  # Utilities
    "200050",  # Information
    "200060",  # Financial activities
    "200070",  # Professional and business services
    "200080",  # Education and health services
    "200090",  # Leisure and hospitality
    "200100",  # Other services except public admin
]

# BLS dataclass codes:
# 01 Gross Job Gains
# 03 Openings
# 04 Gross Job Losses
# 06 Closings
# 07 Establishment Births, if available in quarterly series
# 08 Establishment Deaths, if available in quarterly series
BLS_DATACLASS_CODES = ["01", "03", "04", "06", "07", "08"]

# Full quarterly history. Large file ~250 MB.
BLS_DATA_FILE = "bd.data.1.AllItems"

# U.S. Courts Bankruptcy F-2
BANKRUPTCY_MIN_YEAR = 1990
BANKRUPTCY_MAX_YEAR = 2026

# YC optional
YC_MAX_COMPANIES = 200

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 Safari/605.1.15"
}

session = requests.Session()
session.headers.update(HEADERS)

MANUAL_URLS = []

# ============================================================
# HELPERS
# ============================================================

def now():
    return time.time()


def fmt_seconds(seconds):
    if seconds is None or seconds < 0:
        return "unknown"
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h:
        return f"{h}h {m}m {s}s"
    if m:
        return f"{m}m {s}s"
    return f"{s}s"


def fmt_bytes(n):
    if n is None:
        return "unknown"
    n = float(n)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"


def log_step(msg):
    print("\n" + "=" * 80)
    print(msg)
    print("=" * 80)


def onefile_path(file_name):
    return BASE_DIR / file_name


def add_manual_url(source, url, save_as, reason="Python download failed or may be blocked"):
    row = {
        "source": source,
        "url": url,
        "save_as": str(onefile_path(save_as)),
        "reason": reason,
    }
    MANUAL_URLS.append(row)

    print("\nMANUAL DOWNLOAD NEEDED")
    print("Source:", source)
    print("Reason:", reason)
    print("Open this URL in your browser:")
    print(url)
    print("Save it exactly here:")
    print(onefile_path(save_as))
    print()


def save_manual_url_list():
    out_csv = onefile_path("manual_download_urls.csv")
    out_txt = onefile_path("manual_download_urls.txt")

    if MANUAL_URLS:
        pd.DataFrame(MANUAL_URLS).drop_duplicates().to_csv(out_csv, index=False)
        with open(out_txt, "w", encoding="utf-8") as f:
            for row in MANUAL_URLS:
                f.write("SOURCE: " + row["source"] + "\n")
                f.write("URL: " + row["url"] + "\n")
                f.write("SAVE AS: " + row["save_as"] + "\n")
                f.write("REASON: " + row["reason"] + "\n")
                f.write("-" * 70 + "\n")
        print("Saved manual URL list:", out_csv)
        print("Saved manual URL text:", out_txt)
    else:
        # still create empty files so you know none were needed
        pd.DataFrame(columns=["source", "url", "save_as", "reason"]).to_csv(out_csv, index=False)
        with open(out_txt, "w", encoding="utf-8") as f:
            f.write("No manual downloads needed.\n")


def download_file_or_manual(url, file_name, source, overwrite=False, chunk_size=1024 * 1024):
    """
    Download a file to OneFile/file_name.
    If blocked, print manual URL and return None.
    If user already saved the file manually, it uses that file.
    """
    out_path = onefile_path(file_name)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and out_path.stat().st_size > 0 and not overwrite:
        print(f"Already exists, using: {out_path} ({fmt_bytes(out_path.stat().st_size)})")
        return out_path

    tmp_path = out_path.with_suffix(out_path.suffix + ".part")
    start = now()

    try:
        with session.get(url, stream=True, timeout=180) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0)) or None
            downloaded = 0
            last_print = 0

            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=chunk_size):
                    if not chunk:
                        continue
                    f.write(chunk)
                    downloaded += len(chunk)

                    elapsed = now() - start
                    speed = downloaded / elapsed if elapsed > 0 else 0

                    if total:
                        pct = downloaded / total * 100
                        eta = (total - downloaded) / speed if speed > 0 else None
                        line = (
                            f"\rDownloading {file_name}: {pct:6.2f}% | "
                            f"{fmt_bytes(downloaded)}/{fmt_bytes(total)} | "
                            f"speed {fmt_bytes(speed)}/s | elapsed {fmt_seconds(elapsed)} | ETA {fmt_seconds(eta)}"
                        )
                    else:
                        line = (
                            f"\rDownloading {file_name}: {fmt_bytes(downloaded)} | "
                            f"speed {fmt_bytes(speed)}/s | elapsed {fmt_seconds(elapsed)}"
                        )

                    if now() - last_print > 0.5:
                        print(line, end="")
                        last_print = now()

        tmp_path.replace(out_path)
        print(f"\nSaved: {out_path} | total time: {fmt_seconds(now() - start)}")
        return out_path

    except Exception as e:
        if tmp_path.exists():
            try:
                tmp_path.unlink()
            except Exception:
                pass
        add_manual_url(source, url, file_name, reason=str(e))
        return None


def optimize_dataframe(df):
    for col in df.columns:
        try:
            if pd.api.types.is_integer_dtype(df[col]):
                df[col] = pd.to_numeric(df[col], downcast="integer")
            elif pd.api.types.is_float_dtype(df[col]):
                df[col] = pd.to_numeric(df[col], downcast="float")
            elif df[col].dtype == "object":
                nunique = df[col].nunique(dropna=False)
                total = len(df[col])
                if total > 0 and nunique / total < 0.35:
                    df[col] = df[col].astype("category")
        except Exception:
            pass
    return df


def save_csv_and_parquet(df, csv_name):
    csv_path = onefile_path(csv_name)
    df = optimize_dataframe(df)
    df.to_csv(csv_path, index=False)
    print("Saved CSV:", csv_path)

    try:
        parquet_path = csv_path.with_suffix(".parquet")
        df.to_parquet(parquet_path, index=False)
        print("Saved Parquet:", parquet_path)
    except Exception as e:
        print("Parquet skipped:", e)

    return csv_path


def read_csv_flexible(path, **kwargs):
    return pd.read_csv(path, low_memory=False, keep_default_na=False, **kwargs)


# ============================================================
# PRINT IMPORTANT SOURCE URLS FIRST
# ============================================================

def print_source_urls():
    log_step("SOURCE URLS / MANUAL DOWNLOAD PAGES")
    urls = [
        ("Census BDS source page", "https://www.census.gov/data/datasets/time-series/econ/bds/bds-datasets.html"),
        ("Census BFS monthly CSV", "https://www.census.gov/econ/bfs/csv/bfs_monthly.csv"),
        ("Census BFS month date table", "https://www.census.gov/econ/bfs/csv/month_date_table.csv"),
        ("BLS BED/BDM source page", "https://www.bls.gov/bdm/"),
        ("BLS BED/BDM flat file directory", "https://download.bls.gov/pub/time.series/bd/"),
        ("U.S. Courts caseload data tables", "https://www.uscourts.gov/statistics-reports/caseload-statistics-data-tables"),
        ("U.S. Courts F-2 latest page example", "https://www.uscourts.gov/data-news/data-tables/2025/12/31/bankruptcy-filings/f-2"),
        ("YC startup examples", "https://www.ycombinator.com/companies?regions=United%20States%20of%20America"),
    ]

    for label, url in urls:
        print(f"{label}: {url}")


# ============================================================
# 1. CENSUS BDS
# ============================================================

def scrape_bds_links():
    url = "https://www.census.gov/data/datasets/time-series/econ/bds/bds-datasets.html"
    print("Scraping BDS page:", url)

    try:
        html = session.get(url, timeout=90).text
        soup = BeautifulSoup(html, "html.parser")
    except Exception as e:
        add_manual_url("Census BDS source page", url, "bds_page.html", reason=str(e))
        return pd.DataFrame()

    rows = []
    for a in soup.find_all("a", href=True):
        href = a.get("href", "")
        text = " ".join(a.get_text(" ", strip=True).split())
        if ".csv" in href.lower() and "bds" in href.lower():
            full_url = urljoin(url, href)
            rows.append({
                "source": "Census BDS",
                "name": text if text else Path(full_url).name,
                "url": full_url,
                "file_name": Path(full_url).name,
            })

    links = pd.DataFrame(rows).drop_duplicates(subset=["url"])
    save_csv_and_parquet(links, "bds_download_links.csv")
    print("BDS links found:", len(links))
    return links


def download_and_clean_bds():
    log_step("1. CENSUS BDS: core startup / survival / failure proxy files")

    links = scrape_bds_links()
    if links.empty:
        print("No BDS links found. Use the printed source page manually.")
        return []

    selected = links[links["file_name"].isin(BDS_CORE_FILES)].copy()
    missing_core = [f for f in BDS_CORE_FILES if f not in set(selected["file_name"])]

    print("BDS core files selected:", len(selected))
    if missing_core:
        print("BDS core file names not found on page:")
        for f in missing_core:
            print("-", f)

    clean_index = []

    for _, row in selected.iterrows():
        url = row["url"]
        file_name = row["file_name"]
        table_name = row["name"]

        print("\nBDS table:", table_name)
        raw_path = download_file_or_manual(url, file_name, "Census BDS")
        if raw_path is None:
            continue

        clean_name = "clean_" + file_name
        clean_path = onefile_path(clean_name)

        first = True
        total_rows = 0
        start = now()

        try:
            for chunk_i, chunk in enumerate(pd.read_csv(raw_path, chunksize=200_000, low_memory=False)):
                chunk["source"] = "Census BDS"
                chunk["source_table"] = table_name
                chunk["source_file"] = file_name
                chunk = optimize_dataframe(chunk)

                chunk.to_csv(clean_path, mode="w" if first else "a", index=False, header=first)
                total_rows += len(chunk)
                first = False

                print(
                    f"\rCleaning {file_name}: chunk {chunk_i + 1:,} | rows {total_rows:,} | elapsed {fmt_seconds(now() - start)}",
                    end=""
                )

            print("\nSaved clean BDS:", clean_path)
            clean_index.append({
                "source": "Census BDS",
                "table_name": table_name,
                "raw_file": str(raw_path),
                "clean_file": str(clean_path),
                "rows": total_rows,
            })

        except Exception as e:
            print("\nBDS clean failed:", file_name, e)

        gc.collect()

    index_df = pd.DataFrame(clean_index)
    save_csv_and_parquet(index_df, "bds_clean_file_index.csv")
    return clean_index


# ============================================================
# 2. CENSUS BFS
# ============================================================

def download_and_clean_bfs():
    log_step("2. CENSUS BFS: business applications / new business formation")

    bfs_url = "https://www.census.gov/econ/bfs/csv/bfs_monthly.csv"
    month_url = "https://www.census.gov/econ/bfs/csv/month_date_table.csv"

    bfs_path = download_file_or_manual(bfs_url, "bfs_monthly.csv", "Census BFS")
    month_path = download_file_or_manual(month_url, "month_date_table.csv", "Census BFS")

    if bfs_path is None:
        print("BFS skipped because bfs_monthly.csv is missing.")
        return

    print("Reading BFS:", bfs_path)
    df = read_csv_flexible(bfs_path, dtype=str)
    print("BFS raw shape:", df.shape)
    print("BFS columns:", df.columns.tolist())

    month_cols = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
    existing_month_cols = [c for c in month_cols if c in df.columns]
    id_cols = [c for c in df.columns if c not in existing_month_cols]

    long_df = df.melt(
        id_vars=id_cols,
        value_vars=existing_month_cols,
        var_name="month_name",
        value_name="value_raw"
    )

    month_num_map = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6, "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}
    long_df["month"] = long_df["month_name"].map(month_num_map).astype("Int8")
    long_df["year"] = pd.to_numeric(long_df["year"], errors="coerce").astype("Int16")
    long_df["value"] = (
        long_df["value_raw"].astype(str)
        .str.replace(",", "", regex=False)
        .replace({"NA": None, "D": None, "S": None, "": None})
    )
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df["source"] = "Census BFS"

    if month_path is not None:
        try:
            dates = read_csv_flexible(month_path, dtype=str)
            dates.columns = [c.strip().lower().replace(" ", "_") for c in dates.columns]
            if "year" in dates.columns and "month" in dates.columns:
                dates["year"] = pd.to_numeric(dates["year"], errors="coerce").astype("Int16")
                dates["month"] = pd.to_numeric(dates["month"], errors="coerce").astype("Int8")
                long_df = long_df.merge(dates, on=["year", "month"], how="left")
        except Exception as e:
            print("Month date merge skipped:", e)

    save_csv_and_parquet(long_df, "bfs_monthly_long_clean.csv")

    easy = long_df.copy()
    if "geo" in easy.columns:
        easy = easy[easy["geo"].eq("US")]
    if "sa" in easy.columns:
        easy = easy[easy["sa"].eq("A")]
    if "naics_sector" in easy.columns:
        easy = easy[easy["naics_sector"].eq("TOTAL")]

    easy = easy.sort_values([c for c in ["year", "month", "series"] if c in easy.columns])
    save_csv_and_parquet(easy, "bfs_us_total_seasonally_adjusted.csv")

    del df, long_df, easy
    gc.collect()


# ============================================================
# 3. BLS BED/BDM
# ============================================================

def bls_url(file_name):
    return "https://download.bls.gov/pub/time.series/bd/" + file_name


def print_bls_manual_urls():
    log_step("BLS BED/BDM FILES YOU CAN MANUALLY DOWNLOAD")
    files = [
        "bd.series",
        BLS_DATA_FILE,
        "bd.dataclass",
        "bd.dataelement",
        "bd.industry",
        "bd.ratelevel",
        "bd.state",
        "bd.sizeclass",
        "bd.periodicity",
        "bd.ownership",
        "bd.unitanalysis",
    ]
    for f in files:
        url = bls_url(f)
        print(url)
        print("Save as:", onefile_path(f))


def download_bls_files():
    log_step("3. BLS BED/BDM: openings / closings / job gains / job losses")
    print_bls_manual_urls()

    files = [
        "bd.series",
        BLS_DATA_FILE,
        "bd.dataclass",
        "bd.dataelement",
        "bd.industry",
        "bd.ratelevel",
        "bd.state",
        "bd.sizeclass",
        "bd.periodicity",
        "bd.ownership",
        "bd.unitanalysis",
    ]

    paths = {}
    for f in files:
        paths[f] = download_file_or_manual(bls_url(f), f, "BLS BED/BDM")

    required = ["bd.series", BLS_DATA_FILE]
    missing = [f for f in required if paths.get(f) is None]
    if missing:
        print("\nBLS cannot run yet. Missing required files:")
        for f in missing:
            print("-", f, "download:", bls_url(f), "save as", onefile_path(f))
        return None

    return paths


def read_bls_mapping(path):
    try:
        df = pd.read_csv(path, sep="\t", dtype=str, low_memory=False)
        if df.shape[1] == 1:
            df = pd.read_csv(path, sep=r"\s+", dtype=str, low_memory=False)
    except Exception:
        df = pd.read_csv(path, sep=r"\s+", dtype=str, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    return df


def process_bls_bdm():
    paths = download_bls_files()
    if paths is None:
        return

    series_path = paths["bd.series"]
    data_path = paths[BLS_DATA_FILE]

    log_step("3B. BLS BED/BDM: read and filter series")

    series = read_bls_mapping(series_path)
    print("Series rows:", f"{len(series):,}")
    print("Series columns:", series.columns.tolist())

    needed_cols = [
        "series_id", "seasonal", "state_code", "industry_code", "dataelement_code",
        "sizeclass_code", "dataclass_code", "ratelevel_code", "periodicity_code",
        "ownership_code", "series_title"
    ]
    missing = [c for c in needed_cols if c not in series.columns]
    if missing:
        print("Missing columns in bd.series:", missing)
        series.head(50).to_csv(onefile_path("bls_bdm_series_preview.csv"), index=False)
        print("Saved preview:", onefile_path("bls_bdm_series_preview.csv"))
        return

    selected = series.copy()
    selected = selected[selected["state_code"].isin(BLS_STATE_CODES)]
    selected = selected[selected["dataclass_code"].isin(BLS_DATACLASS_CODES)]
    selected = selected[selected["ratelevel_code"].eq("L")]
    selected = selected[selected["periodicity_code"].eq("Q")]
    selected = selected[selected["ownership_code"].eq("5")]
    selected = selected[selected["sizeclass_code"].eq("00")]

    if BLS_INDUSTRY_CODES is not None:
        selected = selected[selected["industry_code"].isin(BLS_INDUSTRY_CODES)]

    selected_ids = set(selected["series_id"].astype(str))
    print("Selected BLS series:", f"{len(selected):,}")

    selected.to_csv(onefile_path("bls_bdm_selected_series.csv"), index=False)
    print("Saved:", onefile_path("bls_bdm_selected_series.csv"))

    if not selected_ids:
        print("No BLS series selected. Relax filters.")
        return

    log_step("3C. BLS BED/BDM: process big data by chunks")

    # Optional metadata maps
    meta = selected.copy()
    map_specs = [
        ("bd.dataclass", "dataclass_code"),
        ("bd.dataelement", "dataelement_code"),
        ("bd.industry", "industry_code"),
        ("bd.state", "state_code"),
        ("bd.ratelevel", "ratelevel_code"),
    ]

    for f, key in map_specs:
        path = paths.get(f)
        if path is None:
            continue
        try:
            m = read_bls_mapping(path)
            if key in m.columns:
                meta = meta.merge(m, on=key, how="left")
        except Exception as e:
            print("Metadata merge skipped:", f, e)

    keep_meta_cols = [
        "series_id", "seasonal", "state_code", "state_name", "industry_code", "industry_name",
        "dataelement_code", "dataelement_name", "dataclass_code", "dataclass_name",
        "ratelevel_code", "ratelevel_name", "series_title"
    ]
    keep_meta_cols = [c for c in keep_meta_cols if c in meta.columns]
    meta = meta[keep_meta_cols].drop_duplicates("series_id")

    out_path = onefile_path("bls_bdm_openings_closings_job_flows.csv")
    first = True
    total_read = 0
    total_kept = 0
    start = now()

    for chunk_i, chunk in enumerate(pd.read_csv(
        data_path,
        sep=r"\s+",
        dtype={
            "series_id": "string",
            "year": "int16",
            "period": "string",
            "value": "float32",
            "footnote_codes": "string",
        },
        chunksize=1_000_000,
        low_memory=False,
    )):
        chunk.columns = [str(c).strip() for c in chunk.columns]
        total_read += len(chunk)
        chunk = chunk[chunk["series_id"].isin(selected_ids)].copy()

        if len(chunk) > 0:
            chunk = chunk.merge(meta, on="series_id", how="left")
            chunk["quarter"] = chunk["period"].astype(str).str.replace("Q", "", regex=False)
            chunk["quarter"] = pd.to_numeric(chunk["quarter"], errors="coerce").astype("Int8")
            chunk["source"] = "BLS BED/BDM"
            chunk = optimize_dataframe(chunk)

            chunk.to_csv(out_path, mode="w" if first else "a", index=False, header=first)
            total_kept += len(chunk)
            first = False

        elapsed = now() - start
        speed = total_read / elapsed if elapsed > 0 else 0
        print(
            f"\rBLS chunks {chunk_i + 1:,} | read {total_read:,} | kept {total_kept:,} | "
            f"speed {speed:,.0f} rows/s | elapsed {fmt_seconds(elapsed)}",
            end=""
        )

        del chunk
        gc.collect()

    print("\nSaved:", out_path)
    print("Total rows read:", f"{total_read:,}")
    print("Total rows kept:", f"{total_kept:,}")

    if out_path.exists() and total_kept > 0:
        summary_chunks = []
        group_cols = None

        for chunk in pd.read_csv(out_path, chunksize=500_000, low_memory=False):
            possible_cols = [
                "year", "quarter", "state_code", "state_name", "industry_code", "industry_name",
                "dataclass_code", "dataclass_name", "dataelement_code", "dataelement_name"
            ]
            group_cols = [c for c in possible_cols if c in chunk.columns]
            summary_chunks.append(chunk.groupby(group_cols, dropna=False)["value"].sum().reset_index())

        summary = pd.concat(summary_chunks, ignore_index=True)
        summary = summary.groupby(group_cols, dropna=False)["value"].sum().reset_index()
        summary = summary.sort_values(group_cols)
        save_csv_and_parquet(summary, "bls_bdm_summary_by_year_quarter.csv")

        del summary, summary_chunks
        gc.collect()


# ============================================================
# 4. U.S. COURTS BANKRUPTCY F-2
# ============================================================

def scrape_bankruptcy_f2_xlsx_links():
    log_step("4. U.S. Courts Bankruptcy F-2: scrape XLSX links")

    base = "https://www.uscourts.gov/statistics-reports/caseload-statistics-data-tables"
    rows = []
    seen = set()

    # Try pagination. The page structure changes sometimes, so this is intentionally broad.
    for page_num in range(0, 20):
        url = base if page_num == 0 else f"{base}?page={page_num}"
        print("Scraping:", url)
        try:
            html = session.get(url, timeout=90).text
        except Exception as e:
            add_manual_url("U.S. Courts caseload data tables", url, f"uscourts_page_{page_num}.html", reason=str(e))
            continue

        soup = BeautifulSoup(html, "html.parser")
        before = len(rows)

        for tr in soup.find_all("tr"):
            row_text = " ".join(tr.get_text(" ", strip=True).split())
            row_lower = row_text.lower()

            # Keep normal F-2 only. Skip F-2 one-month / three-month.
            is_f2 = re.search(r"\bF-2\b", row_text) is not None
            skip_short_period = "F-2 (" in row_text or "f-2 (" in row_lower
            is_bankruptcy_cases = "business and nonbusiness cases filed" in row_lower

            if not (is_f2 and is_bankruptcy_cases) or skip_short_period:
                continue

            period_match = re.search(
                r"(March|June|September|December)\s+\d{1,2},\s+\d{4}",
                row_text
            )
            period = period_match.group(0) if period_match else ""
            year_match = re.search(r"\b(19|20)\d{2}\b", period)
            year = int(year_match.group(0)) if year_match else None

            if year and not (BANKRUPTCY_MIN_YEAR <= year <= BANKRUPTCY_MAX_YEAR):
                continue

            for a in tr.find_all("a", href=True):
                href = a["href"]
                label = a.get_text(" ", strip=True).lower()
                if ".xlsx" in href.lower() or "xlsx" in label:
                    full_url = urljoin(url, href)
                    if full_url not in seen:
                        seen.add(full_url)
                        rows.append({
                            "source": "U.S. Courts F-2",
                            "reporting_period": period,
                            "year": year,
                            "url": full_url,
                            "file_name": Path(full_url).name,
                            "row_text": row_text,
                        })

        # Stop after a page has no new rows after page 2
        if page_num >= 2 and len(rows) == before:
            break

    # Add known current examples as fallback. These are useful if scraping fails.
    known = [
        ("March 31, 2026", 2026, "https://www.uscourts.gov/sites/default/files/document/bf_f2_0331.2026.xlsx"),
        ("December 31, 2025", 2025, "https://www.uscourts.gov/sites/default/files/document/bf_f2_1231.2025.xlsx"),
        ("December 31, 2025", 2025, "https://www.uscourts.gov/sites/default/files/document/stfj_f2_1231.2025.xlsx"),
    ]

    for period, year, url in known:
        if url not in seen and BANKRUPTCY_MIN_YEAR <= year <= BANKRUPTCY_MAX_YEAR:
            seen.add(url)
            rows.append({
                "source": "U.S. Courts F-2 known fallback",
                "reporting_period": period,
                "year": year,
                "url": url,
                "file_name": Path(url).name,
                "row_text": "known fallback direct URL",
            })

    links = pd.DataFrame(rows).drop_duplicates(subset=["url"])
    if not links.empty:
        links = links.sort_values(["year", "reporting_period", "file_name"], ascending=[False, False, True])

    save_csv_and_parquet(links, "bankruptcy_f2_xlsx_links.csv")
    print("F-2 XLSX links found:", len(links))

    if links.empty:
        add_manual_url(
            "U.S. Courts F-2 source page",
            base,
            "bankruptcy_f2_manual.xlsx",
            reason="No F-2 XLSX links found automatically. Open page and search table F-2."
        )

    return links


def parse_number(x):
    if pd.isna(x):
        return None
    if isinstance(x, (int, float)) and not isinstance(x, bool):
        if math.isnan(x) if isinstance(x, float) else False:
            return None
        return float(x)
    s = str(x).strip().replace(",", "")
    if s in ["", "-", "—", "nan", "None"]:
        return None
    try:
        return float(s)
    except Exception:
        return None


def parse_bankruptcy_xlsx_total_row(path, link_row):
    """
    Robust parser: finds a row labeled Total and saves numeric cells.
    Column names vary across years, so values are saved as value_01, value_02, ...
    """
    try:
        xls = pd.ExcelFile(path)
    except Exception as e:
        return [{
            "source": "U.S. Courts F-2",
            "reporting_period": link_row.get("reporting_period", ""),
            "year": link_row.get("year", None),
            "file_name": Path(path).name,
            "parse_status": "excel_open_error",
            "error": str(e),
        }]

    rows = []
    for sheet in xls.sheet_names:
        try:
            df = pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)
        except Exception:
            continue

        for idx, row in df.iterrows():
            cells = ["" if pd.isna(x) else str(x).strip() for x in row.tolist()]
            row_text = " ".join([c for c in cells if c])
            first_nonempty = next((c for c in cells if c), "")

            # Usually the national row starts with Total.
            if not first_nonempty.lower().startswith("total"):
                continue

            nums = [parse_number(x) for x in row.tolist()]
            nums = [x for x in nums if x is not None]

            if len(nums) < 5:
                continue

            out = {
                "source": "U.S. Courts F-2",
                "reporting_period": link_row.get("reporting_period", ""),
                "year": link_row.get("year", None),
                "file_name": Path(path).name,
                "sheet": sheet,
                "excel_row_number": idx + 1,
                "row_label": first_nonempty,
                "parse_status": "ok",
                "numbers_found": len(nums),
                "raw_total_row_text": row_text,
            }
            for i, val in enumerate(nums, start=1):
                out[f"value_{i:02d}"] = val
            rows.append(out)

    if not rows:
        rows.append({
            "source": "U.S. Courts F-2",
            "reporting_period": link_row.get("reporting_period", ""),
            "year": link_row.get("year", None),
            "file_name": Path(path).name,
            "parse_status": "no_total_row_found",
        })

    return rows


def download_and_clean_bankruptcy_f2():
    links = scrape_bankruptcy_f2_xlsx_links()
    if links.empty:
        print("Bankruptcy skipped because no links were found.")
        return

    all_rows = []

    for _, row in links.iterrows():
        url = row["url"]
        file_name = row["file_name"]
        print("\nBankruptcy F-2:", row.get("reporting_period", ""), file_name)

        path = download_file_or_manual(url, file_name, "U.S. Courts F-2")
        if path is None:
            continue

        parsed_rows = parse_bankruptcy_xlsx_total_row(path, row.to_dict())
        all_rows.extend(parsed_rows)
        print("Parsed rows added:", len(parsed_rows))
        gc.collect()

    if all_rows:
        df = pd.DataFrame(all_rows)
        save_csv_and_parquet(df, "bankruptcy_f2_total_rows_clean.csv")
    else:
        print("No bankruptcy rows parsed yet. Check manual_download_urls.txt.")


# ============================================================
# 5. YC STARTUP EXAMPLES ONLY - OPTIONAL SIMPLE LINK FILE
# ============================================================

def save_yc_example_source():
    log_step("5. YC examples only")
    # This is intentionally simple. YC is examples only, not official success/failure data.
    df = pd.DataFrame([{
        "source": "Y Combinator Directory",
        "note": "Examples only; not official startup survival/failure data",
        "url": "https://www.ycombinator.com/companies?regions=United%20States%20of%20America"
    }])
    save_csv_and_parquet(df, "yc_source_only.csv")


# ============================================================
# FINAL INVENTORY / STATUS
# ============================================================

def create_output_inventory():
    files = []
    for f in sorted(BASE_DIR.glob("*")):
        if f.is_file():
            files.append({
                "file_name": f.name,
                "path": str(f),
                "size": f.stat().st_size,
                "size_readable": fmt_bytes(f.stat().st_size),
            })
    inv = pd.DataFrame(files)
    inv.to_csv(onefile_path("onefile_output_inventory.csv"), index=False)
    return inv


def create_readme():
    readme = onefile_path("README_OneFile.txt")
    text = f"""
ONEFILE BUSINESS STARTUP DATA PROJECT
====================================

Folder:
{BASE_DIR.resolve()}

Main clean outputs to check:
- bds_clean_file_index.csv
- bfs_monthly_long_clean.csv
- bfs_us_total_seasonally_adjusted.csv
- bls_bdm_openings_closings_job_flows.csv
- bls_bdm_summary_by_year_quarter.csv
- bankruptcy_f2_xlsx_links.csv
- bankruptcy_f2_total_rows_clean.csv
- manual_download_urls.txt
- onefile_output_inventory.csv

If BLS BED/BDM is missing:
Open manual_download_urls.txt and download the missing BLS files to the exact OneFile path shown.
Then rerun this script.

If Bankruptcy F-2 is missing:
Open the U.S. Courts caseload page, search for table F-2, download XLSX, save it into OneFile,
and rerun this script.

Source pages:
Census BDS: https://www.census.gov/data/datasets/time-series/econ/bds/bds-datasets.html
Census BFS: https://www.census.gov/econ/bfs/index.html
BLS BED/BDM: https://www.bls.gov/bdm/
BLS flat files: https://download.bls.gov/pub/time.series/bd/
U.S. Courts caseload tables: https://www.uscourts.gov/statistics-reports/caseload-statistics-data-tables
YC examples: https://www.ycombinator.com/companies?regions=United%20States%20of%20America
""".strip()

    with open(readme, "w", encoding="utf-8") as f:
        f.write(text)

    print("Saved README:", readme)


def print_final_status():
    log_step("FINAL STATUS")

    checks = [
        ("BDS clean index", onefile_path("bds_clean_file_index.csv")),
        ("BFS monthly long", onefile_path("bfs_monthly_long_clean.csv")),
        ("BFS easy US total", onefile_path("bfs_us_total_seasonally_adjusted.csv")),
        ("BLS BED/BDM job flows", onefile_path("bls_bdm_openings_closings_job_flows.csv")),
        ("BLS BED/BDM summary", onefile_path("bls_bdm_summary_by_year_quarter.csv")),
        ("Bankruptcy F-2 links", onefile_path("bankruptcy_f2_xlsx_links.csv")),
        ("Bankruptcy F-2 clean", onefile_path("bankruptcy_f2_total_rows_clean.csv")),
        ("Manual URL list", onefile_path("manual_download_urls.txt")),
    ]

    for label, path in checks:
        if path.exists() and path.stat().st_size > 0:
            print(f"READY:   {label} -> {path} ({fmt_bytes(path.stat().st_size)})")
        else:
            print(f"MISSING: {label} -> {path}")

    print("\nAll files in OneFile:")
    for f in sorted(BASE_DIR.glob("*")):
        if f.is_file():
            print(f"- {f.name} | {fmt_bytes(f.stat().st_size)}")


# ============================================================
# RUNNER
# ============================================================

def run_all():
    project_start = now()

    print("PROJECT FOLDER:", BASE_DIR.resolve())
    print_source_urls()

    if RUN_BDS:
        try:
            download_and_clean_bds()
        except Exception as e:
            print("BDS failed:", e)

    if RUN_BFS:
        try:
            download_and_clean_bfs()
        except Exception as e:
            print("BFS failed:", e)

    if RUN_BLS_BDM:
        try:
            process_bls_bdm()
        except Exception as e:
            print("BLS BED/BDM failed:", e)
            print_bls_manual_urls()

    if RUN_BANKRUPTCY:
        try:
            download_and_clean_bankruptcy_f2()
        except Exception as e:
            print("Bankruptcy F-2 failed:", e)
            add_manual_url(
                "U.S. Courts F-2 source page",
                "https://www.uscourts.gov/statistics-reports/caseload-statistics-data-tables",
                "bankruptcy_f2_manual.xlsx",
                reason=str(e)
            )

    if RUN_YC_EXAMPLES:
        try:
            save_yc_example_source()
        except Exception as e:
            print("YC examples failed:", e)

    save_manual_url_list()
    create_readme()
    create_output_inventory()
    print_final_status()

    log_step("DONE")
    print("Total project time:", fmt_seconds(now() - project_start))
    print("Clean outputs are in:", BASE_DIR.resolve())


# ============================================================
# RUN THIS
# ============================================================

run_all()


# 3 Attempt 1 file

In [ ]:
# ============================================================
# ONEFILE BUSINESS STARTUP PROJECT - ONE FINAL CSV VERSION
#
# Final clean output ONLY:
#   OneFile/business_startup_ALL_IN_ONE.csv
#
# Raw downloaded source files are saved in:
#   OneFile/_raw/
#
# Sources:
# 1. Census BDS  = startup / survival / failure proxy
# 2. Census BFS  = business applications / new business formation
# 3. BLS BED/BDM = openings / closings / job gains / job losses
# 4. U.S. Courts F-2 = bankruptcy filings
#
# Fix included:
# - BLS value column is read as text first, so '-' will not crash the code.
# - Everything is converted into one long-format CSV.
# ============================================================

# If packages are missing, run this once in Jupyter:
# !pip install pandas requests beautifulsoup4 openpyxl

import re
import gc
import math
import time
import json
import shutil
import warnings
from pathlib import Path
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path("OneFile")
RAW_DIR = BASE_DIR / "_raw"
BASE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

FINAL_CSV = BASE_DIR / "business_startup_ALL_IN_ONE.csv"

# If True, deletes and recreates the final CSV each run.
OVERWRITE_FINAL_CSV = True

# Turn sources on/off
RUN_BDS = True
RUN_BFS = True
RUN_BLS_BDM = True
RUN_BANKRUPTCY = True

# BDS core files only, not hundreds of huge combinations
BDS_CORE_FILES = [
    "bds2023.csv",          # economy-wide, all years
    "bds2023_st.csv",       # state
    "bds2023_sec.csv",      # sector
    "bds2023_st_sec.csv",   # state by sector
    "bds2023_fa.csv",       # firm age
    "bds2023_st_fa.csv",    # state by firm age
    "bds2023_sec_fa.csv",   # sector by firm age
    "bds2023_ea.csv",       # establishment age
    "bds2023_st_ea.csv",    # state by establishment age
    "bds2023_sec_ea.csv",   # sector by establishment age
]

# BLS BED/BDM filters
# "00" = U.S. total, "15" = Hawaii. Add more if needed: ["00", "15", "06"]
BLS_STATE_CODES = ["00"]

# BLS industries: total private + major sectors
BLS_INDUSTRY_CODES = [
    "000000",  # Total private
    "100000",  # Goods-producing
    "100010",  # Natural resources and mining
    "100020",  # Construction
    "100030",  # Manufacturing
    "200000",  # Service-providing
    "200010",  # Wholesale trade
    "200020",  # Retail trade
    "200030",  # Transportation and warehousing
    "200040",  # Utilities
    "200050",  # Information
    "200060",  # Financial activities
    "200070",  # Professional and business services
    "200080",  # Education and health services
    "200090",  # Leisure and hospitality
    "200100",  # Other services except public admin
]

# BLS dataclass codes:
# 01 Gross Job Gains, 03 Openings, 04 Gross Job Losses, 06 Closings,
# 07 Establishment Births, 08 Establishment Deaths if available
BLS_DATACLASS_CODES = ["01", "03", "04", "06", "07", "08"]

# Full quarterly BED/BDM history. Large file.
BLS_DATA_FILE = "bd.data.1.AllItems"

# Bankruptcy F-2 year range
BANKRUPTCY_MIN_YEAR = 1990
BANKRUPTCY_MAX_YEAR = 2026

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 Safari/605.1.15"
}

session = requests.Session()
session.headers.update(HEADERS)

# ============================================================
# FINAL CSV COMMON SCHEMA
# ============================================================

FINAL_COLUMNS = [
    "source",
    "dataset",
    "source_table",
    "source_file",
    "source_url",
    "year",
    "quarter",
    "month",
    "date",
    "geo_level",
    "geo_code",
    "geo_name",
    "state_code",
    "state_name",
    "county_code",
    "county_name",
    "msa_code",
    "msa_name",
    "metro_nonmetro",
    "naics_code",
    "industry_code",
    "industry_name",
    "firm_age",
    "establishment_age",
    "firm_size",
    "establishment_size",
    "metric_code",
    "metric_name",
    "metric_category",
    "value",
    "unit",
    "seasonal",
    "notes",
    "extra_json",
]

FINAL_HEADER_WRITTEN = False

# ============================================================
# HELPERS
# ============================================================

def now():
    return time.time()


def fmt_seconds(seconds):
    if seconds is None or seconds < 0:
        return "unknown"
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h:
        return f"{h}h {m}m {s}s"
    if m:
        return f"{m}m {s}s"
    return f"{s}s"


def fmt_bytes(n):
    if n is None:
        return "unknown"
    n = float(n)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"


def log_step(msg):
    print("\n" + "=" * 80)
    print(msg)
    print("=" * 80)


def clean_colname(c):
    return str(c).strip().lower().replace(" ", "_").replace("-", "_")


def normalize_columns(df):
    df = df.copy()
    df.columns = [clean_colname(c) for c in df.columns]
    return df


def safe_number_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.strip()
         .str.replace(",", "", regex=False)
         .replace({"": None, "-": None, "—": None, "NA": None, "N/A": None, "nan": None, "None": None}),
        errors="coerce"
    )


def safe_str_value(x):
    if pd.isna(x):
        return ""
    return str(x)


def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def take_col(df, candidates, default=""):
    c = first_existing_col(df, candidates)
    if c is None:
        return pd.Series([default] * len(df), index=df.index)
    return df[c]


def append_to_final(df):
    """Append standardized rows to the one final CSV."""
    global FINAL_HEADER_WRITTEN

    if df is None or len(df) == 0:
        return

    for col in FINAL_COLUMNS:
        if col not in df.columns:
            df[col] = ""

    df = df[FINAL_COLUMNS]

    # Drop rows with missing numeric value unless the value is intentionally text.
    if "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df = df[df["value"].notna()].copy()

    if len(df) == 0:
        return

    mode = "a" if FINAL_HEADER_WRITTEN or FINAL_CSV.exists() else "w"
    header = not (FINAL_HEADER_WRITTEN or FINAL_CSV.exists())
    df.to_csv(FINAL_CSV, mode=mode, header=header, index=False)
    FINAL_HEADER_WRITTEN = True


def find_existing_file(file_name):
    """Use old files if already saved in OneFile root; otherwise use OneFile/_raw."""
    candidates = [
        RAW_DIR / file_name,
        BASE_DIR / file_name,      # supports your previous run where files were in OneFile root
        Path.cwd() / file_name,
        Path.home() / "Downloads" / file_name,
    ]
    for p in candidates:
        if p.exists() and p.stat().st_size > 0:
            if p.parent != RAW_DIR:
                target = RAW_DIR / file_name
                if not target.exists():
                    try:
                        shutil.copy2(p, target)
                        print(f"Found existing file and copied to raw: {p} -> {target}")
                        return target
                    except Exception:
                        print(f"Found existing file, using it: {p}")
                        return p
            print(f"Already exists, using: {p} ({fmt_bytes(p.stat().st_size)})")
            return p
    return None


def download_file_or_manual(url, file_name, source, overwrite=False, chunk_size=1024 * 1024):
    existing = None if overwrite else find_existing_file(file_name)
    if existing is not None:
        return existing

    out_path = RAW_DIR / file_name
    tmp_path = out_path.with_suffix(out_path.suffix + ".part")
    start = now()

    try:
        with session.get(url, stream=True, timeout=180) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0)) or None
            downloaded = 0
            last_print = 0

            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=chunk_size):
                    if not chunk:
                        continue
                    f.write(chunk)
                    downloaded += len(chunk)

                    elapsed = now() - start
                    speed = downloaded / elapsed if elapsed > 0 else 0

                    if total:
                        pct = downloaded / total * 100
                        eta = (total - downloaded) / speed if speed > 0 else None
                        line = (
                            f"\rDownloading {file_name}: {pct:6.2f}% | "
                            f"{fmt_bytes(downloaded)}/{fmt_bytes(total)} | "
                            f"speed {fmt_bytes(speed)}/s | elapsed {fmt_seconds(elapsed)} | ETA {fmt_seconds(eta)}"
                        )
                    else:
                        line = (
                            f"\rDownloading {file_name}: {fmt_bytes(downloaded)} | "
                            f"speed {fmt_bytes(speed)}/s | elapsed {fmt_seconds(elapsed)}"
                        )

                    if now() - last_print > 0.5:
                        print(line, end="")
                        last_print = now()

        tmp_path.replace(out_path)
        print(f"\nSaved raw file: {out_path} | total time: {fmt_seconds(now() - start)}")
        return out_path

    except Exception as e:
        if tmp_path.exists():
            try:
                tmp_path.unlink()
            except Exception:
                pass

        print("\nMANUAL DOWNLOAD NEEDED")
        print("Source:", source)
        print("Open URL:")
        print(url)
        print("Save as:")
        print(out_path)
        print("Reason:", e)
        return None


def print_source_urls():
    log_step("SOURCE URLS")
    urls = [
        ("Census BDS source page", "https://www.census.gov/data/datasets/time-series/econ/bds/bds-datasets.html"),
        ("Census BFS monthly CSV", "https://www.census.gov/econ/bfs/csv/bfs_monthly.csv"),
        ("Census BFS month date table", "https://www.census.gov/econ/bfs/csv/month_date_table.csv"),
        ("BLS BED/BDM source page", "https://www.bls.gov/bdm/"),
        ("BLS BED/BDM flat files", "https://download.bls.gov/pub/time.series/bd/"),
        ("U.S. Courts caseload data tables", "https://www.uscourts.gov/statistics-reports/caseload-statistics-data-tables"),
    ]
    for label, url in urls:
        print(f"{label}: {url}")

# ============================================================
# 1. CENSUS BDS -> one final CSV
# ============================================================

def scrape_bds_links():
    url = "https://www.census.gov/data/datasets/time-series/econ/bds/bds-datasets.html"
    print("Scraping BDS page:", url)

    try:
        html = session.get(url, timeout=90).text
        soup = BeautifulSoup(html, "html.parser")
    except Exception as e:
        print("BDS scrape failed. Open manually:", url)
        print("Reason:", e)
        return pd.DataFrame()

    rows = []
    for a in soup.find_all("a", href=True):
        href = a.get("href", "")
        text = " ".join(a.get_text(" ", strip=True).split())
        if ".csv" in href.lower() and "bds" in href.lower():
            full_url = urljoin(url, href)
            rows.append({
                "name": text if text else Path(full_url).name,
                "url": full_url,
                "file_name": Path(full_url).name,
            })

    return pd.DataFrame(rows).drop_duplicates(subset=["url"])


def bds_dimension_value(row, possible_cols):
    for c in possible_cols:
        if c in row and pd.notna(row[c]):
            return row[c]
    return ""


def process_bds_file(raw_path, source_url, source_table):
    file_name = Path(raw_path).name
    print("Processing BDS into final CSV:", file_name)

    # Columns that are dimensions, not metrics
    dimension_keywords = [
        "year", "yr", "time", "state", "st", "county", "cty", "msa", "metro", "name", "label",
        "sector", "naics", "sic", "fage", "firm_age", "eage", "estab_age", "age",
        "size", "source", "table", "file", "geography", "geo"
    ]

    first_preview = pd.read_csv(raw_path, nrows=5, low_memory=False)
    first_preview = normalize_columns(first_preview)
    all_cols = list(first_preview.columns)

    # Numeric columns that are not obvious dimensions become metrics.
    metric_candidates = []
    for c in all_cols:
        lc = c.lower()
        if any(k in lc for k in dimension_keywords):
            continue
        metric_candidates.append(c)

    if not metric_candidates:
        print("No BDS metric columns found in:", file_name)
        return

    total_out = 0
    start = now()

    for chunk_i, chunk in enumerate(pd.read_csv(raw_path, chunksize=100_000, low_memory=False)):
        chunk = normalize_columns(chunk)

        year_col = first_existing_col(chunk, ["year", "yr"])
        state_code_col = first_existing_col(chunk, ["state", "state_code", "st"])
        state_name_col = first_existing_col(chunk, ["state_name", "stname", "state_label", "name"])
        county_code_col = first_existing_col(chunk, ["county", "county_code", "cty"])
        county_name_col = first_existing_col(chunk, ["county_name", "ctyname"])
        msa_col = first_existing_col(chunk, ["msa", "msa_code", "cbsa"])
        msa_name_col = first_existing_col(chunk, ["msa_name", "cbsa_name"])
        sector_col = first_existing_col(chunk, ["sector", "naics", "naics_code"])
        sector_name_col = first_existing_col(chunk, ["sector_name", "naics_name", "industry_name"])
        firm_age_col = first_existing_col(chunk, ["fage4", "fage", "firm_age", "firm_age_group", "fage4_label"])
        estab_age_col = first_existing_col(chunk, ["eage4", "eage", "establishment_age", "estab_age", "eage4_label"])
        firm_size_col = first_existing_col(chunk, ["fsize", "firm_size", "fz", "ifsize", "ifz"])
        estab_size_col = first_existing_col(chunk, ["esize", "establishment_size", "ez", "iesize", "iez"])
        metro_col = first_existing_col(chunk, ["metro", "metro_nonmetro", "met"])

        available_metric_cols = [c for c in metric_candidates if c in chunk.columns]
        if not available_metric_cols:
            continue

        id_df = pd.DataFrame({
            "year": take_col(chunk, [year_col] if year_col else []),
            "state_code": take_col(chunk, [state_code_col] if state_code_col else []),
            "state_name": take_col(chunk, [state_name_col] if state_name_col else []),
            "county_code": take_col(chunk, [county_code_col] if county_code_col else []),
            "county_name": take_col(chunk, [county_name_col] if county_name_col else []),
            "msa_code": take_col(chunk, [msa_col] if msa_col else []),
            "msa_name": take_col(chunk, [msa_name_col] if msa_name_col else []),
            "industry_code": take_col(chunk, [sector_col] if sector_col else []),
            "industry_name": take_col(chunk, [sector_name_col] if sector_name_col else []),
            "firm_age": take_col(chunk, [firm_age_col] if firm_age_col else []),
            "establishment_age": take_col(chunk, [estab_age_col] if estab_age_col else []),
            "firm_size": take_col(chunk, [firm_size_col] if firm_size_col else []),
            "establishment_size": take_col(chunk, [estab_size_col] if estab_size_col else []),
            "metro_nonmetro": take_col(chunk, [metro_col] if metro_col else []),
        })

        melted = pd.concat([id_df, chunk[available_metric_cols]], axis=1).melt(
            id_vars=list(id_df.columns),
            value_vars=available_metric_cols,
            var_name="metric_name",
            value_name="value"
        )

        melted["value"] = safe_number_series(melted["value"])
        melted = melted[melted["value"].notna()].copy()

        if len(melted) == 0:
            continue

        melted["source"] = "Census BDS"
        melted["dataset"] = "Business Dynamics Statistics"
        melted["source_table"] = source_table
        melted["source_file"] = file_name
        melted["source_url"] = source_url
        melted["metric_code"] = melted["metric_name"]
        melted["metric_category"] = "startup_survival_failure_proxy"
        melted["unit"] = "count_or_rate_from_source"
        melted["geo_level"] = "varies_by_bds_file"
        melted["geo_code"] = melted["state_code"].astype(str)
        melted["geo_name"] = melted["state_name"].astype(str)
        melted["notes"] = "BDS is a startup/survival/failure proxy, not bankruptcy."

        append_to_final(melted)
        total_out += len(melted)

        print(
            f"\rBDS {file_name}: chunk {chunk_i+1:,} | output rows {total_out:,} | elapsed {fmt_seconds(now()-start)}",
            end=""
        )

        del chunk, id_df, melted
        gc.collect()

    print(f"\nBDS done: {file_name} -> rows added {total_out:,}")


def process_bds():
    log_step("1. CENSUS BDS -> FINAL CSV")
    links = scrape_bds_links()
    if links.empty:
        print("No BDS links found. BDS skipped.")
        return

    selected = links[links["file_name"].isin(BDS_CORE_FILES)].copy()
    print("BDS core files selected:", len(selected))

    for _, row in selected.iterrows():
        raw_path = download_file_or_manual(row["url"], row["file_name"], "Census BDS")
        if raw_path is None:
            continue
        process_bds_file(raw_path, row["url"], row["name"])

# ============================================================
# 2. CENSUS BFS -> one final CSV
# ============================================================

def process_bfs():
    log_step("2. CENSUS BFS -> FINAL CSV")

    bfs_url = "https://www.census.gov/econ/bfs/csv/bfs_monthly.csv"
    month_url = "https://www.census.gov/econ/bfs/csv/month_date_table.csv"

    bfs_path = download_file_or_manual(bfs_url, "bfs_monthly.csv", "Census BFS")
    month_path = download_file_or_manual(month_url, "month_date_table.csv", "Census BFS")

    if bfs_path is None:
        print("BFS skipped because bfs_monthly.csv is missing.")
        return

    df = pd.read_csv(bfs_path, dtype=str, low_memory=False, keep_default_na=False)
    df = normalize_columns(df)
    print("BFS raw shape:", df.shape)

    month_cols = [c for c in ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"] if c in df.columns]
    id_cols = [c for c in df.columns if c not in month_cols]

    long_df = df.melt(id_vars=id_cols, value_vars=month_cols, var_name="month_name", value_name="value_raw")

    month_num_map = {
        "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
        "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12
    }
    long_df["month"] = long_df["month_name"].map(month_num_map)
    long_df["year"] = pd.to_numeric(long_df.get("year", ""), errors="coerce")
    long_df["value"] = safe_number_series(long_df["value_raw"])
    long_df = long_df[long_df["value"].notna()].copy()

    # Optional date table merge
    if month_path is not None:
        try:
            dates = pd.read_csv(month_path, dtype=str, low_memory=False, keep_default_na=False)
            dates = normalize_columns(dates)
            if "year" in dates.columns and "month" in dates.columns:
                dates["year"] = pd.to_numeric(dates["year"], errors="coerce")
                dates["month"] = pd.to_numeric(dates["month"], errors="coerce")
                long_df = long_df.merge(dates, on=["year", "month"], how="left", suffixes=("", "_date_table"))
        except Exception as e:
            print("BFS month date merge skipped:", e)

    out = pd.DataFrame({
        "source": "Census BFS",
        "dataset": "Business Formation Statistics",
        "source_table": "bfs_monthly",
        "source_file": Path(bfs_path).name,
        "source_url": bfs_url,
        "year": long_df["year"],
        "month": long_df["month"],
        "date": take_col(long_df, ["date", "month_date", "period", "time_period"], ""),
        "geo_level": "BFS geography",
        "geo_code": take_col(long_df, ["geo"], ""),
        "geo_name": take_col(long_df, ["geo"], ""),
        "industry_code": take_col(long_df, ["naics_sector"], ""),
        "industry_name": take_col(long_df, ["naics_sector"], ""),
        "metric_code": take_col(long_df, ["series"], ""),
        "metric_name": take_col(long_df, ["series"], ""),
        "metric_category": "business_applications_new_business_formation",
        "value": long_df["value"],
        "unit": "count_or_index_from_source",
        "seasonal": take_col(long_df, ["sa"], ""),
        "notes": "BFS monthly data reshaped from wide months to long rows.",
        "extra_json": "",
    })

    append_to_final(out)
    print(f"BFS done -> rows added {len(out):,}")

    del df, long_df, out
    gc.collect()

# ============================================================
# 3. BLS BED/BDM -> one final CSV
# ============================================================

def bls_url(file_name):
    return "https://download.bls.gov/pub/time.series/bd/" + file_name


def print_bls_urls():
    print("\nBLS BED/BDM files. If download fails, manually save these to OneFile/_raw/:")
    for f in [
        "bd.series", BLS_DATA_FILE, "bd.dataclass", "bd.dataelement", "bd.industry",
        "bd.ratelevel", "bd.state", "bd.sizeclass", "bd.periodicity", "bd.ownership", "bd.unitanalysis"
    ]:
        print(bls_url(f))
        print("Save as:", RAW_DIR / f)


def read_bls_mapping(path):
    try:
        df = pd.read_csv(path, sep="\t", dtype=str, low_memory=False, keep_default_na=False)
        if df.shape[1] == 1:
            df = pd.read_csv(path, sep=r"\s+", dtype=str, low_memory=False, keep_default_na=False)
    except Exception:
        df = pd.read_csv(path, sep=r"\s+", dtype=str, low_memory=False, keep_default_na=False)
    df = normalize_columns(df)
    return df


def process_bls_bdm():
    log_step("3. BLS BED/BDM -> FINAL CSV")
    print_bls_urls()

    bls_files = [
        "bd.series", BLS_DATA_FILE, "bd.dataclass", "bd.dataelement", "bd.industry",
        "bd.ratelevel", "bd.state", "bd.sizeclass", "bd.periodicity", "bd.ownership", "bd.unitanalysis"
    ]

    paths = {}
    for f in bls_files:
        paths[f] = download_file_or_manual(bls_url(f), f, "BLS BED/BDM")

    if paths.get("bd.series") is None or paths.get(BLS_DATA_FILE) is None:
        print("BLS skipped because required BLS files are missing.")
        return

    series = read_bls_mapping(paths["bd.series"])
    print("BLS series rows:", f"{len(series):,}")

    needed = [
        "series_id", "state_code", "industry_code", "dataelement_code", "sizeclass_code",
        "dataclass_code", "ratelevel_code", "periodicity_code", "ownership_code", "series_title"
    ]
    missing = [c for c in needed if c not in series.columns]
    if missing:
        print("BLS series missing columns:", missing)
        return

    selected = series.copy()
    selected = selected[selected["state_code"].isin(BLS_STATE_CODES)]
    selected = selected[selected["dataclass_code"].isin(BLS_DATACLASS_CODES)]
    selected = selected[selected["ratelevel_code"].eq("L")]
    selected = selected[selected["periodicity_code"].eq("Q")]
    selected = selected[selected["ownership_code"].eq("5")]
    selected = selected[selected["sizeclass_code"].eq("00")]
    selected = selected[selected["industry_code"].isin(BLS_INDUSTRY_CODES)]

    selected_ids = set(selected["series_id"].astype(str))
    print("Selected BLS series:", f"{len(selected):,}")
    if not selected_ids:
        print("No BLS series selected. BLS skipped.")
        return

    # Merge mapping labels
    meta = selected.copy()
    for f, key in [
        ("bd.dataclass", "dataclass_code"),
        ("bd.dataelement", "dataelement_code"),
        ("bd.industry", "industry_code"),
        ("bd.state", "state_code"),
        ("bd.ratelevel", "ratelevel_code"),
    ]:
        if paths.get(f) is None:
            continue
        try:
            m = read_bls_mapping(paths[f])
            if key in m.columns:
                meta = meta.merge(m, on=key, how="left")
        except Exception as e:
            print("BLS metadata merge skipped:", f, e)

    keep_cols = [
        "series_id", "seasonal", "state_code", "state_name", "industry_code", "industry_name",
        "dataelement_code", "dataelement_name", "dataclass_code", "dataclass_name",
        "ratelevel_code", "ratelevel_name", "series_title"
    ]
    keep_cols = [c for c in keep_cols if c in meta.columns]
    meta = meta[keep_cols].drop_duplicates("series_id")

    total_read = 0
    total_out = 0
    start = now()

    # IMPORTANT FIX: read value as string first, then convert with errors='coerce'. This handles '-'.
    for chunk_i, chunk in enumerate(pd.read_csv(
        paths[BLS_DATA_FILE],
        sep=r"\s+",
        dtype=str,
        chunksize=1_000_000,
        low_memory=False,
        keep_default_na=False,
    )):
        chunk = normalize_columns(chunk)
        total_read += len(chunk)

        if "series_id" not in chunk.columns:
            print("BLS data missing series_id column. Stop.")
            return

        chunk = chunk[chunk["series_id"].isin(selected_ids)].copy()
        if len(chunk) == 0:
            print(
                f"\rBLS chunks {chunk_i+1:,} | read {total_read:,} | output {total_out:,} | elapsed {fmt_seconds(now()-start)}",
                end=""
            )
            continue

        chunk = chunk.merge(meta, on="series_id", how="left")
        chunk["value"] = safe_number_series(chunk.get("value", ""))
        chunk = chunk[chunk["value"].notna()].copy()

        if len(chunk) == 0:
            continue

        chunk["year_num"] = pd.to_numeric(chunk.get("year", ""), errors="coerce")
        chunk["quarter_num"] = pd.to_numeric(chunk.get("period", "").astype(str).str.replace("Q", "", regex=False), errors="coerce")

        out = pd.DataFrame({
            "source": "BLS BED/BDM",
            "dataset": "Business Employment Dynamics / Business Dynamics Measures",
            "source_table": "bd.data.1.AllItems filtered quarterly job flows",
            "source_file": BLS_DATA_FILE,
            "source_url": bls_url(BLS_DATA_FILE),
            "year": chunk["year_num"],
            "quarter": chunk["quarter_num"],
            "geo_level": "state_or_us_total",
            "geo_code": take_col(chunk, ["state_code"], ""),
            "geo_name": take_col(chunk, ["state_name"], ""),
            "state_code": take_col(chunk, ["state_code"], ""),
            "state_name": take_col(chunk, ["state_name"], ""),
            "industry_code": take_col(chunk, ["industry_code"], ""),
            "industry_name": take_col(chunk, ["industry_name"], ""),
            "metric_code": take_col(chunk, ["dataclass_code"], ""),
            "metric_name": take_col(chunk, ["dataclass_name", "series_title"], ""),
            "metric_category": "openings_closings_job_gains_job_losses",
            "value": chunk["value"],
            "unit": take_col(chunk, ["ratelevel_name"], "level"),
            "seasonal": take_col(chunk, ["seasonal"], ""),
            "notes": take_col(chunk, ["series_title"], ""),
            "extra_json": "",
        })

        append_to_final(out)
        total_out += len(out)

        print(
            f"\rBLS chunks {chunk_i+1:,} | read {total_read:,} | output {total_out:,} | elapsed {fmt_seconds(now()-start)}",
            end=""
        )

        del chunk, out
        gc.collect()

    print(f"\nBLS done -> rows added {total_out:,}")

# ============================================================
# 4. U.S. COURTS BANKRUPTCY F-2 -> one final CSV
# ============================================================

def scrape_bankruptcy_f2_links():
    base = "https://www.uscourts.gov/statistics-reports/caseload-statistics-data-tables"
    rows = []
    seen = set()

    for page_num in range(0, 20):
        url = base if page_num == 0 else f"{base}?page={page_num}"
        print("Scraping U.S. Courts:", url)
        try:
            html = session.get(url, timeout=90).text
        except Exception as e:
            print("U.S. Courts scrape failed:", e)
            continue

        soup = BeautifulSoup(html, "html.parser")
        before = len(rows)

        for tr in soup.find_all("tr"):
            row_text = " ".join(tr.get_text(" ", strip=True).split())
            row_lower = row_text.lower()

            is_f2 = re.search(r"\bF-2\b", row_text) is not None
            is_bankruptcy = "business and nonbusiness cases filed" in row_lower
            skip_short = "F-2 (" in row_text or "f-2 (" in row_lower

            if not (is_f2 and is_bankruptcy) or skip_short:
                continue

            period_match = re.search(r"(March|June|September|December)\s+\d{1,2},\s+\d{4}", row_text)
            period = period_match.group(0) if period_match else ""
            year_match = re.search(r"\b(19|20)\d{2}\b", period)
            year = int(year_match.group(0)) if year_match else None

            if year and not (BANKRUPTCY_MIN_YEAR <= year <= BANKRUPTCY_MAX_YEAR):
                continue

            for a in tr.find_all("a", href=True):
                href = a["href"]
                label = a.get_text(" ", strip=True).lower()
                if ".xlsx" in href.lower() or "xlsx" in label:
                    full_url = urljoin(url, href)
                    if full_url not in seen:
                        seen.add(full_url)
                        rows.append({
                            "reporting_period": period,
                            "year": year,
                            "url": full_url,
                            "file_name": Path(full_url).name,
                            "row_text": row_text,
                        })

        if page_num >= 2 and len(rows) == before:
            break

    # Fallback current examples if site structure blocks scraping
    known = [
        ("March 31, 2026", 2026, "https://www.uscourts.gov/sites/default/files/document/bf_f2_0331.2026.xlsx"),
        ("December 31, 2025", 2025, "https://www.uscourts.gov/sites/default/files/document/bf_f2_1231.2025.xlsx"),
        ("December 31, 2025", 2025, "https://www.uscourts.gov/sites/default/files/document/stfj_f2_1231.2025.xlsx"),
    ]
    for period, year, url in known:
        if url not in seen:
            seen.add(url)
            rows.append({
                "reporting_period": period,
                "year": year,
                "url": url,
                "file_name": Path(url).name,
                "row_text": "known fallback direct URL",
            })

    links = pd.DataFrame(rows).drop_duplicates(subset=["url"])
    if not links.empty:
        links = links.sort_values(["year", "reporting_period", "file_name"], ascending=[False, False, True])
    return links


def parse_bankruptcy_xlsx_to_rows(path, link_row):
    rows = []
    try:
        xls = pd.ExcelFile(path)
    except Exception as e:
        print("Could not open bankruptcy Excel:", path, e)
        return rows

    for sheet in xls.sheet_names:
        try:
            df = pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)
        except Exception:
            continue

        for idx, row in df.iterrows():
            cells = ["" if pd.isna(x) else str(x).strip() for x in row.tolist()]
            first_nonempty = next((c for c in cells if c), "")
            if not first_nonempty.lower().startswith("total"):
                continue

            nums = []
            for x in row.tolist():
                n = safe_number_series(pd.Series([x])).iloc[0]
                if pd.notna(n):
                    nums.append(float(n))

            if len(nums) < 5:
                continue

            raw_text = " ".join([c for c in cells if c])
            for i, val in enumerate(nums, start=1):
                rows.append({
                    "source": "U.S. Courts F-2",
                    "dataset": "Bankruptcy filings",
                    "source_table": "F-2 Business and nonbusiness cases filed",
                    "source_file": Path(path).name,
                    "source_url": link_row.get("url", ""),
                    "year": link_row.get("year", ""),
                    "date": link_row.get("reporting_period", ""),
                    "geo_level": "US total",
                    "geo_code": "US",
                    "geo_name": "United States",
                    "metric_code": f"f2_value_{i:02d}",
                    "metric_name": f"F-2 Total row numeric value {i:02d}",
                    "metric_category": "bankruptcy_filings",
                    "value": val,
                    "unit": "filings",
                    "notes": f"Sheet={sheet}; Excel row={idx+1}; Row label={first_nonempty}; Raw row={raw_text}",
                    "extra_json": json.dumps({"numbers_found": len(nums)}, ensure_ascii=False),
                })

    return rows


def process_bankruptcy_f2():
    log_step("4. U.S. COURTS BANKRUPTCY F-2 -> FINAL CSV")

    links = scrape_bankruptcy_f2_links()
    print("F-2 XLSX links found:", len(links))

    if links.empty:
        print("No F-2 links found. Open manually:")
        print("https://www.uscourts.gov/statistics-reports/caseload-statistics-data-tables")
        return

    total_out = 0
    for _, row in links.iterrows():
        raw_path = download_file_or_manual(row["url"], row["file_name"], "U.S. Courts F-2")
        if raw_path is None:
            continue

        parsed_rows = parse_bankruptcy_xlsx_to_rows(raw_path, row.to_dict())
        if parsed_rows:
            out = pd.DataFrame(parsed_rows)
            append_to_final(out)
            total_out += len(out)
            print(f"Parsed bankruptcy rows from {row['file_name']}: {len(out):,}")

        gc.collect()

    print(f"Bankruptcy done -> rows added {total_out:,}")

# ============================================================
# FINAL CHECK
# ============================================================

def final_check():
    log_step("FINAL CHECK")
    if FINAL_CSV.exists() and FINAL_CSV.stat().st_size > 0:
        print("DONE - final one CSV created:")
        print(FINAL_CSV)
        print("Size:", fmt_bytes(FINAL_CSV.stat().st_size))

        try:
            sample = pd.read_csv(FINAL_CSV, nrows=10, low_memory=False)
            print("\nTop 10 rows:")
            print(sample)

            counts = pd.read_csv(FINAL_CSV, usecols=["source"], low_memory=False)["source"].value_counts()
            print("\nRows by source:")
            print(counts)
        except Exception as e:
            print("Could not preview final CSV:", e)
    else:
        print("Final CSV was not created. Check errors above.")


def run_all():
    global FINAL_HEADER_WRITTEN
    project_start = now()

    print("PROJECT FOLDER:", BASE_DIR.resolve())
    print("RAW SOURCE FOLDER:", RAW_DIR.resolve())
    print("FINAL CSV:", FINAL_CSV.resolve())

    if OVERWRITE_FINAL_CSV and FINAL_CSV.exists():
        FINAL_CSV.unlink()
    FINAL_HEADER_WRITTEN = False

    print_source_urls()

    if RUN_BDS:
        try:
            process_bds()
        except Exception as e:
            print("BDS failed:", e)

    if RUN_BFS:
        try:
            process_bfs()
        except Exception as e:
            print("BFS failed:", e)

    if RUN_BLS_BDM:
        try:
            process_bls_bdm()
        except Exception as e:
            print("BLS BED/BDM failed:", e)
            print("This is usually fixed by checking OneFile/_raw/bd.data.1.AllItems exists and rerunning.")

    if RUN_BANKRUPTCY:
        try:
            process_bankruptcy_f2()
        except Exception as e:
            print("Bankruptcy F-2 failed:", e)

    final_check()

    log_step("DONE")
    print("Total project time:", fmt_seconds(now() - project_start))
    print("Only final clean CSV output:", FINAL_CSV.resolve())


# ============================================================
# RUN THIS
# ============================================================

run_all()


# Fix stuff

In [ ]:
from pathlib import Path
import pandas as pd
import json
import gc
import time

# ============================================================
# FIX BLS BED/BDM AND APPEND INTO EXISTING ONE FINAL CSV
# ============================================================

BASE_DIR = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile"
RAW_DIR = BASE_DIR / "_raw"
FINAL_CSV = BASE_DIR / "business_startup_ALL_IN_ONE.csv"

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("FINAL_CSV:", FINAL_CSV)
print()

if not RAW_DIR.exists():
    raise FileNotFoundError(f"Missing raw folder: {RAW_DIR}")

if not FINAL_CSV.exists():
    raise FileNotFoundError(f"Missing final CSV: {FINAL_CSV}")

# BLS files needed
needed_files = [
    "bd.series",
    "bd.data.1.AllItems",
    "bd.dataclass",
    "bd.dataelement",
    "bd.industry",
    "bd.ratelevel",
    "bd.state",
]

missing = [f for f in needed_files if not (RAW_DIR / f).exists()]
if missing:
    print("Missing BLS raw files:")
    for f in missing:
        print("-", f)
    raise FileNotFoundError("Download missing BLS files first.")
else:
    print("All needed BLS raw files found.")
    for f in needed_files:
        p = RAW_DIR / f
        print(f"- {f}: {p.stat().st_size:,} bytes")

print()


# ============================================================
# SETTINGS
# ============================================================

BLS_STATE_CODES = ["00"]   # "00" = U.S. total

BLS_INDUSTRY_CODES = [
    "000000",  # Total private
    "100000",  # Goods-producing
    "100010",  # Natural resources and mining
    "100020",  # Construction
    "100030",  # Manufacturing
    "200000",  # Service-providing
    "200010",  # Wholesale trade
    "200020",  # Retail trade
    "200030",  # Transportation and warehousing
    "200040",  # Utilities
    "200050",  # Information
    "200060",  # Financial activities
    "200070",  # Professional and business services
    "200080",  # Education and health services
    "200090",  # Leisure and hospitality
    "200100",  # Other services except public admin
]

BLS_DATACLASS_CODES = ["01", "03", "04", "06", "07", "08"]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_string_columns(df):
    df.columns = [str(c).strip() for c in df.columns]
    for c in df.columns:
        if df[c].dtype == "object" or str(df[c].dtype).startswith("string"):
            df[c] = df[c].astype(str).str.strip()
            df[c] = df[c].replace({"nan": "", "None": "", "<NA>": ""})
    return df


def normalize_series_id(s):
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\s+", "", regex=True)
    )


def safe_number_series(s):
    return pd.to_numeric(
        s.astype(str)
        .str.strip()
        .str.replace(",", "", regex=False)
        .replace({"": None, "-": None, "—": None, "nan": None, "None": None}),
        errors="coerce"
    )


def read_bls_table(path):
    # BLS metadata files are usually tab separated.
    df = pd.read_csv(path, sep="\t", dtype=str, low_memory=False)
    if df.shape[1] == 1:
        df = pd.read_csv(path, sep=r"\s+", dtype=str, low_memory=False)
    return clean_string_columns(df)


def make_name_map(path, code_col, desired_name):
    try:
        df = read_bls_table(path)
        if code_col not in df.columns:
            return pd.DataFrame(columns=[code_col, desired_name])

        possible_name_cols = [
            desired_name,
            code_col.replace("_code", "_name"),
            code_col.replace("_code", "_text"),
        ]

        name_col = None
        for c in possible_name_cols:
            if c in df.columns and c != code_col:
                name_col = c
                break

        if name_col is None:
            other_cols = [c for c in df.columns if c != code_col]
            if other_cols:
                name_col = other_cols[0]

        if name_col is None:
            return pd.DataFrame(columns=[code_col, desired_name])

        out = df[[code_col, name_col]].copy()
        out = out.rename(columns={name_col: desired_name})
        out = out.drop_duplicates(subset=[code_col])
        return out

    except Exception as e:
        print(f"Mapping skipped for {path.name}: {e}")
        return pd.DataFrame(columns=[code_col, desired_name])


# ============================================================
# STEP 1: READ AND FILTER BLS SERIES
# ============================================================

series_path = RAW_DIR / "bd.series"
data_path = RAW_DIR / "bd.data.1.AllItems"

series = read_bls_table(series_path)

print("bd.series rows:", len(series))
print("bd.series columns:")
print(series.columns.tolist())
print()

for col in [
    "series_id", "state_code", "industry_code", "dataclass_code",
    "ratelevel_code", "periodicity_code", "ownership_code", "sizeclass_code"
]:
    if col in series.columns:
        series[col] = series[col].astype(str).str.strip()

series["series_id_clean"] = normalize_series_id(series["series_id"])

selected = series.copy()

selected = selected[selected["state_code"].isin(BLS_STATE_CODES)]
selected = selected[selected["industry_code"].isin(BLS_INDUSTRY_CODES)]
selected = selected[selected["dataclass_code"].isin(BLS_DATACLASS_CODES)]
selected = selected[selected["ratelevel_code"].eq("L")]
selected = selected[selected["periodicity_code"].eq("Q")]
selected = selected[selected["ownership_code"].eq("5")]
selected = selected[selected["sizeclass_code"].eq("00")]

selected_ids = set(selected["series_id_clean"].dropna().astype(str))

print("Selected BLS series:", len(selected_ids))

if len(selected_ids) == 0:
    raise ValueError("No BLS series selected. Filters are too strict or bd.series format changed.")

print("Example selected series IDs:")
print(selected[["series_id", "series_id_clean", "series_title"]].head(10).to_string(index=False))
print()


# ============================================================
# STEP 2: ADD METADATA NAMES
# ============================================================

meta = selected.copy()

maps = [
    (RAW_DIR / "bd.dataclass", "dataclass_code", "dataclass_name"),
    (RAW_DIR / "bd.dataelement", "dataelement_code", "dataelement_name"),
    (RAW_DIR / "bd.industry", "industry_code", "industry_name"),
    (RAW_DIR / "bd.state", "state_code", "state_name"),
    (RAW_DIR / "bd.ratelevel", "ratelevel_code", "ratelevel_name"),
]

for path, code_col, name_col in maps:
    m = make_name_map(path, code_col, name_col)
    if len(m) > 0 and code_col in meta.columns:
        meta = meta.merge(m, on=code_col, how="left")

meta_keep = [
    "series_id_clean",
    "seasonal",
    "state_code",
    "state_name",
    "industry_code",
    "industry_name",
    "dataelement_code",
    "dataelement_name",
    "dataclass_code",
    "dataclass_name",
    "ratelevel_code",
    "ratelevel_name",
    "series_title",
]

meta_keep = [c for c in meta_keep if c in meta.columns]
meta = meta[meta_keep].drop_duplicates(subset=["series_id_clean"])

print("Metadata rows:", len(meta))
print()


# ============================================================
# STEP 3: CHECK FINAL CSV COLUMNS
# ============================================================

final_columns = pd.read_csv(FINAL_CSV, nrows=0).columns.tolist()

print("Final CSV columns:", len(final_columns))
print(final_columns)
print()


# ============================================================
# STEP 4: REMOVE OLD BLS ROWS IF THEY EXIST
# This prevents duplicates if you run this more than once.
# ============================================================

print("Checking if final CSV already has BLS rows...")

source_counts = {}
for chunk in pd.read_csv(FINAL_CSV, usecols=["source"], chunksize=500_000, low_memory=False):
    counts = chunk["source"].value_counts(dropna=False).to_dict()
    for k, v in counts.items():
        source_counts[k] = source_counts.get(k, 0) + v

print("Current rows by source:")
for k, v in source_counts.items():
    print(f"- {k}: {v:,}")

has_bls = any(str(k).strip() == "BLS BED/BDM" for k in source_counts)

if has_bls:
    print("\nExisting BLS rows found. Rewriting CSV without old BLS rows first...")
    temp_no_bls = FINAL_CSV.with_name("business_startup_ALL_IN_ONE_no_old_bls.tmp.csv")

    first = True
    kept = 0
    for chunk in pd.read_csv(FINAL_CSV, chunksize=300_000, low_memory=False):
        chunk = chunk[chunk["source"].astype(str).str.strip() != "BLS BED/BDM"]
        chunk.to_csv(temp_no_bls, mode="w" if first else "a", index=False, header=first)
        kept += len(chunk)
        first = False
        print(f"\rRows kept without old BLS: {kept:,}", end="")

    print()
    temp_no_bls.replace(FINAL_CSV)
    print("Old BLS rows removed.")
else:
    print("No old BLS rows found. Good.")


# ============================================================
# STEP 5: PROCESS BIG BLS DATA AND APPEND TO FINAL CSV
# ============================================================

print("\nProcessing BLS big data and appending into final CSV...")
start = time.time()

total_read = 0
total_kept = 0
total_output_rows = 0
first_debug_done = False

for chunk_i, chunk in enumerate(pd.read_csv(
    data_path,
    sep=r"\s+",
    dtype=str,
    chunksize=1_000_000,
    low_memory=False,
)):
    chunk = clean_string_columns(chunk)

    if "series_id" not in chunk.columns:
        print("BLS data chunk columns:", chunk.columns.tolist())
        raise ValueError("BLS data file does not have series_id column.")

    total_read += len(chunk)
    chunk["series_id_clean"] = normalize_series_id(chunk["series_id"])

    if not first_debug_done:
        data_sample_ids = set(chunk["series_id_clean"].head(5000).astype(str))
        sample_common = data_sample_ids.intersection(selected_ids)
        print("Debug first chunk:")
        print("- first chunk rows:", len(chunk))
        print("- sample selected id:", next(iter(selected_ids)))
        print("- common IDs in first 5,000 data rows:", len(sample_common))
        first_debug_done = True

    chunk = chunk[chunk["series_id_clean"].isin(selected_ids)].copy()
    total_kept += len(chunk)

    if len(chunk) > 0:
        chunk = chunk.merge(meta, on="series_id_clean", how="left")

        chunk["value"] = safe_number_series(chunk["value"])
        chunk = chunk.dropna(subset=["value"])

        chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce").astype("Int64")
        chunk["quarter"] = (
            chunk["period"]
            .astype(str)
            .str.replace("Q", "", regex=False)
            .pipe(pd.to_numeric, errors="coerce")
            .astype("Int64")
        )

        out = pd.DataFrame()
        out["source"] = "BLS BED/BDM"
        out["dataset"] = "Business Employment Dynamics"
        out["source_table"] = "BED/BDM quarterly job flows"
        out["source_file"] = "bd.data.1.AllItems"
        out["source_url"] = "https://download.bls.gov/pub/time.series/bd/bd.data.1.AllItems"

        out["year"] = chunk["year"]
        out["quarter"] = chunk["quarter"]

        if "month" in final_columns:
            out["month"] = pd.NA
        if "date" in final_columns:
            out["date"] = pd.NA

        out["geo_level"] = "state_or_us"
        out["state_code"] = chunk.get("state_code", "")
        out["state_name"] = chunk.get("state_name", "")
        out["industry_code"] = chunk.get("industry_code", "")
        out["industry_name"] = chunk.get("industry_name", "")

        # Optional dimension columns used by BDS/BFS
        for c in ["naics_sector", "series", "firm_age", "establishment_age", "firm_size", "establishment_size"]:
            if c in final_columns and c not in out.columns:
                out[c] = pd.NA

        out["metric_code"] = chunk.get("dataclass_code", "")
        out["metric_name"] = chunk.get("dataclass_name", chunk.get("series_title", ""))
        out["metric_category"] = "job_flows_openings_closings_gains_losses"
        out["value"] = chunk["value"]
        out["unit"] = "level_from_source"
        out["seasonal"] = chunk.get("seasonal", "")
        out["notes"] = "BLS BED/BDM quarterly data: job gains, openings, job losses, closings, births/deaths where available."

        extra_cols = [
            "series_id",
            "series_id_clean",
            "period",
            "dataelement_code",
            "dataelement_name",
            "ratelevel_code",
            "ratelevel_name",
            "series_title",
            "footnote_codes",
        ]

        extra = chunk[[c for c in extra_cols if c in chunk.columns]].astype(str)
        out["extra_json"] = extra.apply(lambda r: json.dumps(r.to_dict(), ensure_ascii=False), axis=1)

        # Match the exact final CSV columns
        for c in final_columns:
            if c not in out.columns:
                out[c] = pd.NA

        out = out[final_columns]

        out.to_csv(FINAL_CSV, mode="a", index=False, header=False)
        total_output_rows += len(out)

    elapsed = time.time() - start
    speed = total_read / elapsed if elapsed > 0 else 0

    print(
        f"\rChunk {chunk_i + 1:,} | read {total_read:,} | matched {total_kept:,} | "
        f"output {total_output_rows:,} | speed {speed:,.0f} rows/s | elapsed {elapsed:,.0f}s",
        end=""
    )

    del chunk
    gc.collect()

print()
print("\nBLS processing finished.")
print("Total BLS rows read:", f"{total_read:,}")
print("Total BLS rows matched:", f"{total_kept:,}")
print("Total BLS rows added to final CSV:", f"{total_output_rows:,}")

if total_output_rows == 0:
    print("\nStill 0 BLS rows. Need to inspect series_id format.")
    raise ValueError("BLS matched 0 rows. Send me the debug output above.")
else:
    print("\nSUCCESS: BLS rows were added to your final one CSV.")


# ============================================================
# STEP 6: FINAL CHECK
# ============================================================

print("\nFinal check rows by source:")
final_counts = {}

for chunk in pd.read_csv(FINAL_CSV, usecols=["source"], chunksize=500_000, low_memory=False):
    counts = chunk["source"].value_counts(dropna=False).to_dict()
    for k, v in counts.items():
        final_counts[k] = final_counts.get(k, 0) + v

for k, v in final_counts.items():
    print(f"- {k}: {v:,}")

print()
print("DONE. Your final CSV is still:")
print(FINAL_CSV)
print("Size:", f"{FINAL_CSV.stat().st_size / 1024 / 1024:.2f} MB")

In [ ]:
#Check the path
from pathlib import Path

BASE_DIR = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile"
RAW_DIR = BASE_DIR / "_raw"
FINAL_CSV = BASE_DIR / "business_startup_ALL_IN_ONE.csv"

print("BASE_DIR exists:", BASE_DIR.exists(), BASE_DIR)
print("RAW_DIR exists:", RAW_DIR.exists(), RAW_DIR)
print("FINAL_CSV exists:", FINAL_CSV.exists(), FINAL_CSV)

print()
print("BLS big file exists:", (RAW_DIR / "bd.data.1.AllItems").exists())
print("BLS series file exists:", (RAW_DIR / "bd.series").exists())

# Check for missing or nan

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import time

# ============================================================
# DATA QUALITY CHECK FOR business_startup_ALL_IN_ONE.csv
# Memory optimized: chunk read, no full file load
# ============================================================

BASE_DIR = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile"
FINAL_CSV = BASE_DIR / "business_startup_ALL_IN_ONE.csv"
REPORT_CSV = BASE_DIR / "business_startup_DATA_QUALITY_REPORT.csv"
BAD_ROWS_SAMPLE_CSV = BASE_DIR / "business_startup_BAD_ROWS_SAMPLE.csv"

print("Checking file:")
print(FINAL_CSV)
print()

if not FINAL_CSV.exists():
    raise FileNotFoundError(f"Missing final CSV: {FINAL_CSV}")

print("File exists.")
print("Size MB:", round(FINAL_CSV.stat().st_size / 1024 / 1024, 2))
print()

# ============================================================
# Expected structure
# ============================================================

required_columns = [
    "source",
    "dataset",
    "source_table",
    "source_file",
    "source_url",
    "year",
    "metric_code",
    "metric_name",
    "metric_category",
    "value",
    "unit",
]

expected_sources = {
    "Census BDS",
    "Census BFS",
    "U.S. Courts F-2",
    "BLS BED/BDM",
}

# Some columns are allowed to be blank depending on source:
# BDS = yearly, so quarter/month/date can be blank.
# BFS = monthly, so quarter can be blank.
# BLS = quarterly, so month/date can be blank.
# Bankruptcy = special Excel table, many dimension columns can be blank.

allowed_blank_columns = {
    "quarter",
    "month",
    "date",
    "state_code",
    "state_name",
    "industry_code",
    "industry_name",
    "naics_sector",
    "series",
    "firm_age",
    "establishment_age",
    "firm_size",
    "establishment_size",
    "seasonal",
    "notes",
    "extra_json",
}

# ============================================================
# Header check
# ============================================================

header = pd.read_csv(FINAL_CSV, nrows=0)
columns = header.columns.tolist()

print("Column count:", len(columns))
print("Columns:")
print(columns)
print()

missing_required_columns = [c for c in required_columns if c not in columns]
duplicate_columns = pd.Series(columns).value_counts()
duplicate_columns = duplicate_columns[duplicate_columns > 1].to_dict()

if missing_required_columns:
    print("ERROR - missing required columns:")
    print(missing_required_columns)
else:
    print("Required columns: OK")

if duplicate_columns:
    print("ERROR - duplicate columns:")
    print(duplicate_columns)
else:
    print("Duplicate columns: OK")

print()

# ============================================================
# Chunk checks
# ============================================================

total_rows = 0
source_counts = {}
metric_category_counts = {}
bad_source_counts = {}
critical_missing_counts = {c: 0 for c in required_columns}
nan_counts = {c: 0 for c in columns}
blank_counts = {c: 0 for c in columns}

bad_year_count = 0
bad_value_count = 0
negative_value_count = 0
bad_quarter_count = 0
bad_month_count = 0
bad_extra_json_count = 0

min_year = None
max_year = None

bad_rows_sample = []
max_bad_sample = 500

start = time.time()

for chunk_i, chunk in enumerate(pd.read_csv(FINAL_CSV, chunksize=300_000, low_memory=False)):
    total_rows += len(chunk)

    # Normalize important string columns
    for c in ["source", "dataset", "metric_category", "metric_code", "metric_name"]:
        if c in chunk.columns:
            chunk[c] = chunk[c].astype(str).str.strip()

    # Source counts
    if "source" in chunk.columns:
        vc = chunk["source"].value_counts(dropna=False)
        for k, v in vc.items():
            source_counts[str(k)] = source_counts.get(str(k), 0) + int(v)

        bad_sources = chunk[~chunk["source"].isin(expected_sources)]
        if len(bad_sources) > 0:
            vc_bad = bad_sources["source"].value_counts(dropna=False)
            for k, v in vc_bad.items():
                bad_source_counts[str(k)] = bad_source_counts.get(str(k), 0) + int(v)

    # Metric category counts
    if "metric_category" in chunk.columns:
        vc = chunk["metric_category"].value_counts(dropna=False)
        for k, v in vc.items():
            metric_category_counts[str(k)] = metric_category_counts.get(str(k), 0) + int(v)

    # NaN / blank counts
    for c in columns:
        s = chunk[c]
        nan_counts[c] += int(s.isna().sum())

        if s.dtype == "object":
            blank_counts[c] += int(s.astype(str).str.strip().isin(["", "nan", "None", "<NA>"]).sum())
        else:
            blank_counts[c] += int(s.isna().sum())

    # Critical missing counts
    for c in required_columns:
        if c in chunk.columns:
            s = chunk[c]
            if s.dtype == "object":
                missing = s.isna() | s.astype(str).str.strip().isin(["", "nan", "None", "<NA>"])
            else:
                missing = s.isna()
            critical_missing_counts[c] += int(missing.sum())

    # Year check
    if "year" in chunk.columns:
        year_num = pd.to_numeric(chunk["year"], errors="coerce")
        valid_year = year_num.dropna()

        if len(valid_year) > 0:
            y_min = int(valid_year.min())
            y_max = int(valid_year.max())
            min_year = y_min if min_year is None else min(min_year, y_min)
            max_year = y_max if max_year is None else max(max_year, y_max)

        bad_year = year_num.isna() | (year_num < 1900) | (year_num > 2100)
        bad_year_count += int(bad_year.sum())
    else:
        bad_year = pd.Series([True] * len(chunk), index=chunk.index)

    # Value check
    if "value" in chunk.columns:
        value_num = pd.to_numeric(chunk["value"], errors="coerce")
        bad_value = value_num.isna()
        bad_value_count += int(bad_value.sum())
        negative_value_count += int((value_num < 0).sum())
    else:
        bad_value = pd.Series([True] * len(chunk), index=chunk.index)

    # Quarter check
    if "quarter" in chunk.columns:
        q = pd.to_numeric(chunk["quarter"], errors="coerce")
        # quarter can be blank for non-BLS sources, but if present must be 1-4
        bad_q = q.notna() & ~q.isin([1, 2, 3, 4])
        bad_quarter_count += int(bad_q.sum())
    else:
        bad_q = pd.Series([False] * len(chunk), index=chunk.index)

    # Month check
    if "month" in chunk.columns:
        m = pd.to_numeric(chunk["month"], errors="coerce")
        # month can be blank for non-BFS sources, but if present must be 1-12
        bad_m = m.notna() & ~m.between(1, 12)
        bad_month_count += int(bad_m.sum())
    else:
        bad_m = pd.Series([False] * len(chunk), index=chunk.index)

    # extra_json check, only when it is not blank
    if "extra_json" in chunk.columns:
        ej = chunk["extra_json"].astype(str)
        maybe_json = ~ej.str.strip().isin(["", "nan", "None", "<NA>"])

        # Check only a small sample per chunk to stay fast
        sample_json = ej[maybe_json].head(1000)
        for x in sample_json:
            try:
                json.loads(x)
            except Exception:
                bad_extra_json_count += 1

    # Save sample bad rows
    bad_any = bad_year | bad_value | bad_q | bad_m

    if len(bad_rows_sample) < max_bad_sample and bad_any.any():
        sample = chunk.loc[bad_any].head(max_bad_sample - len(bad_rows_sample)).copy()
        bad_rows_sample.append(sample)

    elapsed = time.time() - start
    print(
        f"\rChunk {chunk_i + 1:,} | rows checked {total_rows:,} | "
        f"elapsed {elapsed:,.0f}s",
        end=""
    )

print()
print()

# ============================================================
# Build report
# ============================================================

report_rows = []

def add_report(check_name, status, value, detail=""):
    report_rows.append({
        "check_name": check_name,
        "status": status,
        "value": value,
        "detail": detail,
    })

add_report("file_exists", "PASS", str(FINAL_CSV.exists()), str(FINAL_CSV))
add_report("file_size_mb", "INFO", round(FINAL_CSV.stat().st_size / 1024 / 1024, 2), "")
add_report("total_rows", "INFO", total_rows, "")
add_report("column_count", "INFO", len(columns), "")
add_report(
    "required_columns",
    "PASS" if not missing_required_columns else "FAIL",
    len(required_columns) - len(missing_required_columns),
    "Missing: " + ", ".join(missing_required_columns) if missing_required_columns else "All required columns found"
)
add_report(
    "duplicate_columns",
    "PASS" if not duplicate_columns else "FAIL",
    len(duplicate_columns),
    str(duplicate_columns)
)
add_report("min_year", "INFO", min_year, "")
add_report("max_year", "INFO", max_year, "")
add_report("bad_year_count", "PASS" if bad_year_count == 0 else "WARN", bad_year_count, "Year should be 1900-2100 and not blank")
add_report("bad_value_count", "PASS" if bad_value_count == 0 else "WARN", bad_value_count, "Value should be numeric and not blank")
add_report("negative_value_count", "INFO", negative_value_count, "Negative may be okay for some change metrics, but review if unexpected")
add_report("bad_quarter_count", "PASS" if bad_quarter_count == 0 else "WARN", bad_quarter_count, "Quarter must be 1-4 when present")
add_report("bad_month_count", "PASS" if bad_month_count == 0 else "WARN", bad_month_count, "Month must be 1-12 when present")
add_report("bad_extra_json_sample_count", "PASS" if bad_extra_json_count == 0 else "WARN", bad_extra_json_count, "Only sampled nonblank extra_json")

# Source checks
for src in expected_sources:
    count = source_counts.get(src, 0)
    status = "PASS" if count > 0 else "FAIL"
    add_report(f"source_present_{src}", status, count, "")

if bad_source_counts:
    add_report("unexpected_source_values", "WARN", sum(bad_source_counts.values()), str(bad_source_counts))
else:
    add_report("unexpected_source_values", "PASS", 0, "")

# Critical missing report
for c, cnt in critical_missing_counts.items():
    status = "PASS" if cnt == 0 else "WARN"
    add_report(f"critical_missing_{c}", status, cnt, "")

# General blank counts
for c in columns:
    cnt = blank_counts[c]
    pct = round(cnt / total_rows * 100, 2) if total_rows else 0

    if c in allowed_blank_columns:
        status = "INFO"
        detail = "Allowed to be blank for some sources"
    elif c in required_columns:
        status = "PASS" if cnt == 0 else "WARN"
        detail = "Required column"
    else:
        status = "INFO"
        detail = ""

    add_report(f"blank_count_{c}", status, cnt, f"{pct}% blank. {detail}")

# Counts by source
for src, cnt in sorted(source_counts.items()):
    add_report(f"rows_by_source_{src}", "INFO", cnt, "")

# Counts by metric category
for cat, cnt in sorted(metric_category_counts.items()):
    add_report(f"rows_by_metric_category_{cat}", "INFO", cnt, "")

report_df = pd.DataFrame(report_rows)
report_df.to_csv(REPORT_CSV, index=False)

print("Saved report:")
print(REPORT_CSV)

if bad_rows_sample:
    bad_df = pd.concat(bad_rows_sample, ignore_index=True)
    bad_df.to_csv(BAD_ROWS_SAMPLE_CSV, index=False)
    print("Saved bad rows sample:")
    print(BAD_ROWS_SAMPLE_CSV)
else:
    print("No bad rows sample created because no bad rows were found.")

print()
print("SUMMARY")
print("=======")
print("Total rows:", f"{total_rows:,}")
print("Year range:", min_year, "-", max_year)
print()
print("Rows by source:")
for k, v in sorted(source_counts.items()):
    print(f"- {k}: {v:,}")

print()
print("Important checks:")
print("- Required columns missing:", missing_required_columns if missing_required_columns else "None")
print("- Bad year count:", f"{bad_year_count:,}")
print("- Bad value count:", f"{bad_value_count:,}")
print("- Bad quarter count:", f"{bad_quarter_count:,}")
print("- Bad month count:", f"{bad_month_count:,}")
print("- Unexpected source values:", bad_source_counts if bad_source_counts else "None")

print()
if source_counts.get("BLS BED/BDM", 0) == 0:
    print("WARNING: BLS BED/BDM is still missing from the final CSV.")
else:
    print("BLS BED/BDM is included.")

if bad_value_count == 0 and bad_year_count == 0 and not missing_required_columns:
    print("MAIN CHECK PASSED: critical columns look good.")
else:
    print("MAIN CHECK HAS WARNINGS: open the report CSV and bad row sample.")

# Fix 2

In [ ]:
from pathlib import Path
import pandas as pd
import shutil
import time

# ============================================================
# FIX source = nan rows in final CSV
# Most likely these are BLS BED/BDM rows with missing source label
# ============================================================

BASE_DIR = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile"
FINAL_CSV = BASE_DIR / "business_startup_ALL_IN_ONE.csv"
FIXED_CSV = BASE_DIR / "business_startup_ALL_IN_ONE_fixed.tmp.csv"

print("FINAL_CSV:", FINAL_CSV)
print("Exists:", FINAL_CSV.exists())
print("Size MB:", round(FINAL_CSV.stat().st_size / 1024 / 1024, 2))
print()

# ============================================================
# STEP 1: Inspect rows where source is blank/nan
# ============================================================

print("Inspecting source = nan rows...")

sample_rows = []
source_counts = {}
nan_source_count = 0

for chunk in pd.read_csv(FINAL_CSV, chunksize=300_000, low_memory=False):
    source_str = chunk["source"].astype(str).str.strip()
    is_nan_source = chunk["source"].isna() | source_str.isin(["", "nan", "None", "<NA>"])

    nan_source_count += int(is_nan_source.sum())

    if is_nan_source.any() and len(sample_rows) < 20:
        sample = chunk.loc[
            is_nan_source,
            [
                "source",
                "dataset",
                "source_table",
                "source_file",
                "year",
                "quarter",
                "metric_code",
                "metric_name",
                "metric_category",
                "value",
                "extra_json",
            ]
        ].head(20 - len(sample_rows))
        sample_rows.append(sample)

print("Rows with source nan/blank:", f"{nan_source_count:,}")

if sample_rows:
    sample_df = pd.concat(sample_rows, ignore_index=True)
    print()
    print("Sample source = nan rows:")
    print(sample_df.to_string(index=False))
else:
    print("No nan source rows found.")

print()

# ============================================================
# STEP 2: Rewrite CSV and label those rows as BLS BED/BDM
# Only fixes rows that look like BLS:
# - source_file is bd.data.1.AllItems, OR
# - dataset is Business Employment Dynamics, OR
# - metric_category is job_flows_openings_closings_gains_losses
# ============================================================

print("Fixing source labels...")

first = True
total_rows = 0
fixed_rows = 0
start = time.time()

for chunk_i, chunk in enumerate(pd.read_csv(FINAL_CSV, chunksize=300_000, low_memory=False)):
    total_rows += len(chunk)

    source_str = chunk["source"].astype(str).str.strip()
    is_blank_source = chunk["source"].isna() | source_str.isin(["", "nan", "None", "<NA>"])

    looks_like_bls = pd.Series(False, index=chunk.index)

    if "source_file" in chunk.columns:
        looks_like_bls = looks_like_bls | chunk["source_file"].astype(str).str.strip().eq("bd.data.1.AllItems")

    if "dataset" in chunk.columns:
        looks_like_bls = looks_like_bls | chunk["dataset"].astype(str).str.strip().eq("Business Employment Dynamics")

    if "metric_category" in chunk.columns:
        looks_like_bls = looks_like_bls | chunk["metric_category"].astype(str).str.strip().eq("job_flows_openings_closings_gains_losses")

    fix_mask = is_blank_source & looks_like_bls
    fixed_rows += int(fix_mask.sum())

    chunk.loc[fix_mask, "source"] = "BLS BED/BDM"

    # Optional: fill other BLS labels if blank
    if "dataset" in chunk.columns:
        blank_dataset = chunk["dataset"].isna() | chunk["dataset"].astype(str).str.strip().isin(["", "nan", "None", "<NA>"])
        chunk.loc[fix_mask & blank_dataset, "dataset"] = "Business Employment Dynamics"

    if "source_table" in chunk.columns:
        blank_table = chunk["source_table"].isna() | chunk["source_table"].astype(str).str.strip().isin(["", "nan", "None", "<NA>"])
        chunk.loc[fix_mask & blank_table, "source_table"] = "BED/BDM quarterly job flows"

    if "metric_category" in chunk.columns:
        blank_cat = chunk["metric_category"].isna() | chunk["metric_category"].astype(str).str.strip().isin(["", "nan", "None", "<NA>"])
        chunk.loc[fix_mask & blank_cat, "metric_category"] = "job_flows_openings_closings_gains_losses"

    chunk.to_csv(FIXED_CSV, mode="w" if first else "a", index=False, header=first)
    first = False

    elapsed = time.time() - start
    print(
        f"\rChunk {chunk_i + 1:,} | checked {total_rows:,} | fixed {fixed_rows:,} | elapsed {elapsed:,.0f}s",
        end=""
    )

print()
print("Total rows checked:", f"{total_rows:,}")
print("Rows fixed as BLS BED/BDM:", f"{fixed_rows:,}")

if fixed_rows == 0:
    print()
    print("No rows were fixed. The nan rows may not be BLS rows.")
    print("Do NOT replace the file yet. Send me the sample printed above.")
else:
    # Replace original with fixed version
    backup = BASE_DIR / "business_startup_ALL_IN_ONE_before_source_fix.csv"

    if not backup.exists():
        print("Creating backup:")
        print(backup)
        shutil.copy2(FINAL_CSV, backup)
    else:
        print("Backup already exists:")
        print(backup)

    FIXED_CSV.replace(FINAL_CSV)

    print()
    print("DONE - fixed file replaced original:")
    print(FINAL_CSV)
    print("Backup saved as:")
    print(backup)

In [ ]:
from pathlib import Path
import pandas as pd
import time

# ============================================================
# FINAL QUALITY CHECK AFTER BLS SOURCE FIX
# ============================================================

BASE_DIR = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile"
FINAL_CSV = BASE_DIR / "business_startup_ALL_IN_ONE.csv"
REPORT_CSV = BASE_DIR / "business_startup_FINAL_CHECK_REPORT.csv"

print("Checking file:")
print(FINAL_CSV)
print()

if not FINAL_CSV.exists():
    raise FileNotFoundError(f"Missing file: {FINAL_CSV}")

print("File exists.")
print("Size MB:", round(FINAL_CSV.stat().st_size / 1024 / 1024, 2))
print()

# Expected sources
expected_sources = {
    "Census BDS",
    "Census BFS",
    "BLS BED/BDM",
    "U.S. Courts F-2",
}

required_columns = [
    "source",
    "dataset",
    "source_table",
    "source_file",
    "source_url",
    "year",
    "metric_code",
    "metric_name",
    "metric_category",
    "value",
    "unit",
]

# Read header only
header = pd.read_csv(FINAL_CSV, nrows=0)
columns = header.columns.tolist()

print("Column count:", len(columns))
print("Columns:")
print(columns)
print()

missing_required = [c for c in required_columns if c not in columns]
duplicate_columns = pd.Series(columns).value_counts()
duplicate_columns = duplicate_columns[duplicate_columns > 1].to_dict()

print("Required columns:", "OK" if not missing_required else missing_required)
print("Duplicate columns:", "OK" if not duplicate_columns else duplicate_columns)
print()

# Chunk checks
total_rows = 0
source_counts = {}
metric_category_counts = {}

bad_source_counts = {}
missing_required_counts = {c: 0 for c in required_columns}

bad_year_count = 0
bad_value_count = 0
bad_quarter_count = 0
bad_month_count = 0

min_year = None
max_year = None

start = time.time()

for chunk_i, chunk in enumerate(pd.read_csv(FINAL_CSV, chunksize=300_000, low_memory=False)):
    total_rows += len(chunk)

    # Clean source text
    chunk["source_clean"] = chunk["source"].astype(str).str.strip()

    # Source counts
    vc = chunk["source_clean"].value_counts(dropna=False)
    for k, v in vc.items():
        source_counts[k] = source_counts.get(k, 0) + int(v)

    # Unexpected source
    bad_sources = chunk[~chunk["source_clean"].isin(expected_sources)]
    if len(bad_sources) > 0:
        bad_vc = bad_sources["source_clean"].value_counts(dropna=False)
        for k, v in bad_vc.items():
            bad_source_counts[k] = bad_source_counts.get(k, 0) + int(v)

    # Metric category counts
    if "metric_category" in chunk.columns:
        vc = chunk["metric_category"].astype(str).str.strip().value_counts(dropna=False)
        for k, v in vc.items():
            metric_category_counts[k] = metric_category_counts.get(k, 0) + int(v)

    # Required missing counts
    for c in required_columns:
        if c in chunk.columns:
            s = chunk[c]
            missing = s.isna() | s.astype(str).str.strip().isin(["", "nan", "None", "<NA>"])
            missing_required_counts[c] += int(missing.sum())

    # Year check
    year_num = pd.to_numeric(chunk["year"], errors="coerce")
    valid_year = year_num.dropna()

    if len(valid_year) > 0:
        y_min = int(valid_year.min())
        y_max = int(valid_year.max())
        min_year = y_min if min_year is None else min(min_year, y_min)
        max_year = y_max if max_year is None else max(max_year, y_max)

    bad_year = year_num.isna() | (year_num < 1900) | (year_num > 2100)
    bad_year_count += int(bad_year.sum())

    # Value check
    value_num = pd.to_numeric(chunk["value"], errors="coerce")
    bad_value_count += int(value_num.isna().sum())

    # Quarter check: blank allowed, but if present must be 1-4
    if "quarter" in chunk.columns:
        q = pd.to_numeric(chunk["quarter"], errors="coerce")
        bad_q = q.notna() & ~q.isin([1, 2, 3, 4])
        bad_quarter_count += int(bad_q.sum())

    # Month check: blank allowed, but if present must be 1-12
    if "month" in chunk.columns:
        m = pd.to_numeric(chunk["month"], errors="coerce")
        bad_m = m.notna() & ~m.between(1, 12)
        bad_month_count += int(bad_m.sum())

    elapsed = time.time() - start
    print(f"\rChunk {chunk_i + 1:,} | rows checked {total_rows:,} | elapsed {elapsed:,.0f}s", end="")

print()
print()

# Save report
report_rows = []

def add_report(check, status, value, detail=""):
    report_rows.append({
        "check": check,
        "status": status,
        "value": value,
        "detail": detail,
    })

add_report("total_rows", "INFO", total_rows)
add_report("year_range", "INFO", f"{min_year} - {max_year}")
add_report("required_columns_missing", "PASS" if not missing_required else "FAIL", str(missing_required))
add_report("duplicate_columns", "PASS" if not duplicate_columns else "FAIL", str(duplicate_columns))
add_report("bad_year_count", "PASS" if bad_year_count == 0 else "WARN", bad_year_count)
add_report("bad_value_count", "PASS" if bad_value_count == 0 else "WARN", bad_value_count)
add_report("bad_quarter_count", "PASS" if bad_quarter_count == 0 else "WARN", bad_quarter_count)
add_report("bad_month_count", "PASS" if bad_month_count == 0 else "WARN", bad_month_count)
add_report("unexpected_source_values", "PASS" if not bad_source_counts else "WARN", str(bad_source_counts))

for src in expected_sources:
    count = source_counts.get(src, 0)
    add_report(f"source_present_{src}", "PASS" if count > 0 else "FAIL", count)

for c, count in missing_required_counts.items():
    add_report(f"missing_required_{c}", "PASS" if count == 0 else "WARN", count)

for src, count in sorted(source_counts.items()):
    add_report(f"rows_by_source_{src}", "INFO", count)

for cat, count in sorted(metric_category_counts.items()):
    add_report(f"rows_by_metric_category_{cat}", "INFO", count)

report_df = pd.DataFrame(report_rows)
report_df.to_csv(REPORT_CSV, index=False)

print("Saved report:")
print(REPORT_CSV)
print()

print("SUMMARY")
print("=======")
print("Total rows:", f"{total_rows:,}")
print("Year range:", min_year, "-", max_year)
print()

print("Rows by source:")
for k, v in sorted(source_counts.items()):
    print(f"- {k}: {v:,}")

print()
print("Important checks:")
print("- Required columns missing:", missing_required if missing_required else "None")
print("- Bad year count:", f"{bad_year_count:,}")
print("- Bad value count:", f"{bad_value_count:,}")
print("- Bad quarter count:", f"{bad_quarter_count:,}")
print("- Bad month count:", f"{bad_month_count:,}")
print("- Unexpected source values:", bad_source_counts if bad_source_counts else "None")

print()
if source_counts.get("BLS BED/BDM", 0) > 0:
    print("BLS BED/BDM is included.")
else:
    print("WARNING: BLS BED/BDM is still missing.")

if (
    not missing_required
    and not duplicate_columns
    and bad_year_count == 0
    and bad_value_count == 0
    and bad_quarter_count == 0
    and bad_month_count == 0
    and not bad_source_counts
    and source_counts.get("BLS BED/BDM", 0) > 0
):
    print("MAIN CHECK PASSED: final CSV looks good.")
else:
    print("MAIN CHECK HAS WARNINGS: check the report.")

# show report

In [ ]:
from pathlib import Path
import pandas as pd

FINAL_CSV = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile" / "business_startup_ALL_IN_ONE.csv"

# Show all columns in output
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

print("File:")
print(FINAL_CSV)
print()

# Print column names
cols = pd.read_csv(FINAL_CSV, nrows=0).columns.tolist()

print("Total columns:", len(cols))
print("Columns:")
for i, col in enumerate(cols, start=1):
    print(f"{i}. {col}")

print("\nTop 20 rows:")
df20 = pd.read_csv(FINAL_CSV, nrows=20, low_memory=False)
display(df20)

In [ ]:
from pathlib import Path
import pandas as pd

FINAL_CSV = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile" / "business_startup_ALL_IN_ONE.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

sources = ["Census BDS", "Census BFS", "BLS BED/BDM", "U.S. Courts F-2"]

for src in sources:
    print("\n" + "=" * 80)
    print(f"Top 20 rows for: {src}")
    print("=" * 80)

    rows = []
    for chunk in pd.read_csv(FINAL_CSV, chunksize=200_000, low_memory=False):
        part = chunk[chunk["source"].astype(str).str.strip().eq(src)]
        if len(part) > 0:
            rows.append(part)
        if sum(len(x) for x in rows) >= 20:
            break

    if rows:
        display(pd.concat(rows, ignore_index=True).head(20))
    else:
        print("No rows found.")

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# MEMORY-OPTIMIZED: SHOW COLUMNS + TOP 20 ROWS
# ============================================================

FINAL_CSV = (
    Path.home()
    / "Downloads"
    / "Internship_SCIPE CI-SIP"
    / "MainProject"
    / "3_Company"
    / "OneFile"
    / "business_startup_ALL_IN_ONE.csv"
)

print("File:")
print(FINAL_CSV)
print()

if not FINAL_CSV.exists():
    raise FileNotFoundError(f"File not found: {FINAL_CSV}")

print("File exists.")
print("Size MB:", round(FINAL_CSV.stat().st_size / 1024 / 1024, 2))
print()

# Show all columns and wide output
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 25)

# Read only header first
columns = pd.read_csv(FINAL_CSV, nrows=0).columns.tolist()

print("Total columns:", len(columns))
print()
print("Columns:")
for i, col in enumerate(columns, start=1):
    print(f"{i}. {col}")

print()
print("=" * 100)
print("TOP 20 ROWS")
print("=" * 100)

# Read only 20 rows = memory optimized
df20 = pd.read_csv(FINAL_CSV, nrows=20, low_memory=False)

print("Rows loaded:", len(df20))
print("Columns loaded:", len(df20.columns))
print()

display(df20)

# Double Check nan

In [ ]:
from pathlib import Path
import pandas as pd

file_path = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile" / "business_startup_ALL_IN_ONE.csv"

print("Checking file:")
print(file_path)

if not file_path.exists():
    raise FileNotFoundError(f"File not found: {file_path}")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 80)

df = pd.read_csv(file_path, nrows=20)

# Only for display: replace NaN with blank
df_display = df.fillna("")

print("Rows:", len(df_display))
print("Columns:", len(df_display.columns))
print()

display(df_display)

In [ ]:
from pathlib import Path
import pandas as pd

file_path = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile" / "business_startup_ALL_IN_ONE.csv"

important_cols = [
    "source",
    "dataset",
    "source_table",
    "source_file",
    "year",
    "metric_code",
    "metric_name",
    "metric_category",
    "value",
    "unit"
]

df = pd.read_csv(file_path, nrows=100000)

print("Missing values in important columns:")
print(df[important_cols].isna().sum())

# Search test

In [ ]:
from pathlib import Path
import pandas as pd

file_path = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile" / "business_startup_ALL_IN_ONE.csv"

YEAR = 2010
CHUNKSIZE = 200_000

# Chemical Engineer related industry keywords
keywords = [
    "chemical",
    "pharmaceutical",
    "petroleum",
    "plastics",
    "rubber",
    "scientific research",
    "research and development",
    "biotechnology"
]

# Common related NAICS prefixes
# 325 = Chemical Manufacturing
# 324 = Petroleum and Coal Products Manufacturing
# 326 = Plastics and Rubber Products Manufacturing
# 5417 = Scientific Research and Development Services
related_naics_prefixes = ("325", "324", "326", "5417")

wanted_metric_keywords = [
    "firm",
    "establishment",
    "birth",
    "death",
    "opening",
    "closing",
    "startup",
    "job_gain",
    "job_loss"
]

print("Checking file:")
print(file_path)

if not file_path.exists():
    raise FileNotFoundError(file_path)

matches = []
total_rows = 0

for chunk_num, chunk in enumerate(pd.read_csv(file_path, chunksize=CHUNKSIZE, low_memory=False), start=1):
    total_rows += len(chunk)

    # Keep only selected year
    chunk = chunk[chunk["year"] == YEAR].copy()
    if chunk.empty:
        continue

    # Make safe string columns
    for col in ["industry_name", "naics_code", "metric_code", "metric_name", "source", "dataset", "source_table"]:
        if col not in chunk.columns:
            chunk[col] = ""
        chunk[col] = chunk[col].fillna("").astype(str)

    industry_lower = chunk["industry_name"].str.lower()
    metric_text = (chunk["metric_code"] + " " + chunk["metric_name"]).str.lower()

    keyword_mask = False
    for kw in keywords:
        keyword_mask = keyword_mask | industry_lower.str.contains(kw, regex=False)

    naics_mask = chunk["naics_code"].str.startswith(related_naics_prefixes)

    metric_mask = False
    for kw in wanted_metric_keywords:
        metric_mask = metric_mask | metric_text.str.contains(kw, regex=False)

    result = chunk[(keyword_mask | naics_mask) & metric_mask].copy()

    if not result.empty:
        matches.append(result)

    if chunk_num % 10 == 0:
        print(f"Checked chunk {chunk_num:,} | rows checked {total_rows:,}")

if matches:
    df = pd.concat(matches, ignore_index=True)

    keep_cols = [
        "source", "dataset", "source_table", "year",
        "geo_level", "geo_name", "state_name", "county_name", "msa_name",
        "naics_code", "industry_name",
        "metric_code", "metric_name", "metric_category",
        "value", "unit", "notes"
    ]

    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].copy()

    df = df.sort_values(["source", "industry_name", "metric_code"], na_position="last")

    print()
    print("FOUND Chemical Engineer-related company/employer proxy rows")
    print("Rows found:", len(df))
    print()

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 300)
    pd.set_option("display.max_colwidth", 80)

    display(df.head(100).fillna(""))

    output_path = file_path.parent / f"chemical_engineer_related_company_proxy_{YEAR}.csv"
    df.to_csv(output_path, index=False)

    print()
    print("Saved:")
    print(output_path)

else:
    print()
    print("No Chemical Engineer-related industry/company proxy rows found for year:", YEAR)

# Test 2

In [ ]:
from pathlib import Path
import pandas as pd

file_path = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile" / "business_startup_ALL_IN_ONE.csv"

YEAR = 2010
CHUNKSIZE = 200_000

total_2010 = 0
has_naics = 0
has_industry = 0
examples = []

for i, chunk in enumerate(pd.read_csv(file_path, chunksize=CHUNKSIZE, low_memory=False), start=1):
    chunk["year_num"] = pd.to_numeric(chunk["year"], errors="coerce")
    y = chunk[chunk["year_num"] == YEAR].copy()

    if y.empty:
        continue

    total_2010 += len(y)

    for col in ["naics_code", "industry_name"]:
        if col not in y.columns:
            y[col] = ""

    y["naics_code"] = y["naics_code"].fillna("").astype(str).str.strip()
    y["industry_name"] = y["industry_name"].fillna("").astype(str).str.strip()

    has_naics += (y["naics_code"] != "").sum()
    has_industry += (y["industry_name"] != "").sum()

    sample = y[
        (y["naics_code"] != "") | 
        (y["industry_name"] != "")
    ].head(20)

    if not sample.empty:
        examples.append(sample)

    if i % 10 == 0:
        print(f"Checked chunk {i:,}")

print("Year:", YEAR)
print("Total rows for year:", total_2010)
print("Rows with NAICS code:", has_naics)
print("Rows with industry name:", has_industry)

if examples:
    df_examples = pd.concat(examples, ignore_index=True)
    display(df_examples.head(50).fillna(""))
else:
    print()
    print("No industry-level rows found for this year.")

# Search Test 3

In [ ]:
from pathlib import Path
import pandas as pd

file_path = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile" / "business_startup_ALL_IN_ONE.csv"

YEAR = 2010
CHUNKSIZE = 200_000

rows = []

for chunk in pd.read_csv(file_path, chunksize=CHUNKSIZE, low_memory=False):
    chunk["year_num"] = pd.to_numeric(chunk["year"], errors="coerce")
    y = chunk[chunk["year_num"] == YEAR].copy()

    if y.empty:
        continue

    for col in ["source", "dataset", "source_table", "industry_code", "industry_name", "metric_code", "metric_name"]:
        if col not in y.columns:
            y[col] = ""

    y["industry_code"] = y["industry_code"].fillna("").astype(str).str.strip()
    y["industry_name"] = y["industry_name"].fillna("").astype(str).str.strip()

    y = y[(y["industry_code"] != "") | (y["industry_name"] != "")]

    if not y.empty:
        rows.append(y[[
            "source", "dataset", "source_table",
            "industry_code", "industry_name",
            "metric_code", "metric_name"
        ]])

if rows:
    df = pd.concat(rows, ignore_index=True).drop_duplicates()
    df = df.sort_values(["source", "dataset", "source_table", "industry_code", "industry_name"])

    print("Unique industry rows for", YEAR)
    print("Rows:", len(df))
    display(df.head(200).fillna(""))
else:
    print("No industry rows found.")

In [ ]:
from pathlib import Path
import pandas as pd
import time

# ============================================================
# CAREER / DEGREE SEARCH CONFIG
# ============================================================

QUERY = "Chemical Engineer"
YEAR = 2010

COMPANY_FILE = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile" / "business_startup_ALL_IN_ONE.csv"

CHUNKSIZE = 200_000

# ============================================================
# SEARCH MAP
# This is how degree/job search connects to company/business proxy.
# ============================================================

SEARCH_MAP = {
    "chemical engineer": {
        "display_name": "Chemical Engineer",

        # Degree side
        "degree_cip_codes": ["14.0701"],
        "degree_keywords": [
            "chemical engineering",
            "chemical engineer"
        ],

        # Occupation side
        "soc_codes": ["17-2041"],
        "occupation_keywords": [
            "chemical engineers",
            "chemical engineer"
        ],

        # Company/business side
        # Stronger industry proxy if your file has detailed NAICS.
        "strong_industry_codes": [
            "325",      # Chemical Manufacturing
            "324",      # Petroleum and Coal Products Manufacturing
            "326",      # Plastics and Rubber Products Manufacturing
            "5417",     # Scientific Research and Development Services
            "541330",   # Engineering Services
        ],

        "strong_industry_keywords": [
            "chemical",
            "pharmaceutical",
            "petroleum",
            "plastics",
            "rubber",
            "scientific research",
            "research and development",
            "biotechnology",
            "engineering services"
        ],

        # Broad proxy if your file only has sectors.
        # Your file appears to have broad sector codes like NAICSMNF and NAICS54.
        "broad_industry_codes": [
            "31-33",      # Manufacturing
            "31", "32", "33",
            "NAICSMNF",   # BFS Manufacturing
            "NAICS31",
            "NAICS32",
            "NAICS33",
            "54",
            "NAICS54",    # Professional, Scientific, and Technical Services
        ],

        "broad_industry_keywords": [
            "manufacturing",
            "professional, scientific",
            "scientific and technical",
            "scientific",
            "technical services"
        ],
    }
}

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_search_config(query):
    q = query.lower().strip()

    if q in SEARCH_MAP:
        return SEARCH_MAP[q]

    # fallback simple search
    return {
        "display_name": query,
        "degree_cip_codes": [],
        "degree_keywords": [q],
        "soc_codes": [],
        "occupation_keywords": [q],
        "strong_industry_codes": [],
        "strong_industry_keywords": [q],
        "broad_industry_codes": [],
        "broad_industry_keywords": []
    }


def safe_string_series(df, col):
    if col not in df.columns:
        return pd.Series([""] * len(df), index=df.index)
    return df[col].fillna("").astype(str).str.strip()


def build_company_proxy_mask(chunk, config):
    industry_code = safe_string_series(chunk, "industry_code")
    industry_name = safe_string_series(chunk, "industry_name")
    naics_code = safe_string_series(chunk, "naics_code")

    code_text = (
        industry_code + " " + 
        industry_name + " " + 
        naics_code
    ).str.upper()

    name_text = (
        industry_name + " " + 
        industry_code + " " + 
        naics_code
    ).str.lower()

    strong_mask = pd.Series(False, index=chunk.index)
    broad_mask = pd.Series(False, index=chunk.index)

    # Strong code match
    for code in config["strong_industry_codes"]:
        code_upper = code.upper()

        strong_mask = strong_mask | code_text.str.contains(code_upper, regex=False)

        # Also match if code begins with detailed NAICS
        strong_mask = strong_mask | industry_code.str.upper().str.startswith(code_upper)
        strong_mask = strong_mask | naics_code.str.upper().str.startswith(code_upper)

    # Strong keyword match
    for kw in config["strong_industry_keywords"]:
        strong_mask = strong_mask | name_text.str.contains(kw.lower(), regex=False)

    # Broad code match
    for code in config["broad_industry_codes"]:
        code_upper = code.upper()

        broad_mask = broad_mask | code_text.str.contains(code_upper, regex=False)
        broad_mask = broad_mask | industry_code.str.upper().str.startswith(code_upper)
        broad_mask = broad_mask | naics_code.str.upper().str.startswith(code_upper)

    # Broad keyword match
    for kw in config["broad_industry_keywords"]:
        broad_mask = broad_mask | name_text.str.contains(kw.lower(), regex=False)

    return strong_mask, broad_mask


def search_company_proxy(company_file, query, year, chunksize=200_000):
    config = get_search_config(query)

    print("=" * 90)
    print("CAREER SEARCH")
    print("=" * 90)
    print("Search:", config["display_name"])
    print("Year:", year)
    print()
    print("Degree connection:")
    print("  CIP codes:", config["degree_cip_codes"])
    print("  Degree keywords:", config["degree_keywords"])
    print()
    print("Occupation connection:")
    print("  SOC codes:", config["soc_codes"])
    print("  Occupation keywords:", config["occupation_keywords"])
    print()
    print("Company/business connection:")
    print("  Exact company count by occupation/degree: NOT AVAILABLE DIRECTLY")
    print("  Using related industry proxy.")
    print("=" * 90)
    print()

    if not company_file.exists():
        raise FileNotFoundError(f"Company file not found: {company_file}")

    all_matches = []
    start_time = time.time()
    rows_checked = 0

    for chunk_num, chunk in enumerate(pd.read_csv(company_file, chunksize=chunksize, low_memory=False), start=1):
        rows_checked += len(chunk)

        if "year" not in chunk.columns:
            raise ValueError("Missing column: year")

        chunk["year_num"] = pd.to_numeric(chunk["year"], errors="coerce")
        y = chunk[chunk["year_num"] == year].copy()

        if y.empty:
            continue

        strong_mask, broad_mask = build_company_proxy_mask(y, config)

        strong_rows = y[strong_mask].copy()
        broad_rows = y[(~strong_mask) & broad_mask].copy()

        if not strong_rows.empty:
            strong_rows["match_type"] = "strong_related_industry_proxy"
            all_matches.append(strong_rows)

        if not broad_rows.empty:
            broad_rows["match_type"] = "broad_related_industry_proxy"
            all_matches.append(broad_rows)

        if chunk_num % 10 == 0:
            elapsed = round(time.time() - start_time, 1)
            print(f"Checked chunk {chunk_num:,} | rows checked {rows_checked:,} | elapsed {elapsed}s")

    print()
    print("Finished scanning.")
    print("Rows checked:", f"{rows_checked:,}")

    if not all_matches:
        print()
        print("No related company/business proxy rows found.")
        return None, None

    df = pd.concat(all_matches, ignore_index=True)

    # Make value numeric
    if "value" in df.columns:
        df["value_num"] = pd.to_numeric(df["value"], errors="coerce")
    else:
        df["value_num"] = pd.NA

    # Keep useful columns
    keep_cols = [
        "match_type",
        "source",
        "dataset",
        "source_table",
        "source_file",
        "year",
        "quarter",
        "month",
        "geo_level",
        "geo_code",
        "geo_name",
        "state_code",
        "state_name",
        "county_code",
        "county_name",
        "msa_code",
        "msa_name",
        "naics_code",
        "industry_code",
        "industry_name",
        "metric_code",
        "metric_name",
        "metric_category",
        "value",
        "unit",
        "seasonal",
        "notes"
    ]

    keep_cols = [c for c in keep_cols if c in df.columns]
    detail_df = df[keep_cols].copy()

    # Summary group
    group_cols = [
        "match_type",
        "source",
        "dataset",
        "source_table",
        "year",
        "geo_level",
        "geo_name",
        "state_name",
        "industry_code",
        "industry_name",
        "metric_code",
        "metric_name",
        "metric_category",
        "unit"
    ]

    group_cols = [c for c in group_cols if c in df.columns]

    summary_df = (
        df.groupby(group_cols, dropna=False)
          .agg(
              total_value=("value_num", "sum"),
              rows=("value_num", "size"),
              non_missing_values=("value_num", "count")
          )
          .reset_index()
          .sort_values(["match_type", "source", "industry_code", "metric_code"], na_position="last")
    )

    # Save outputs
    safe_query = query.lower().replace(" ", "_").replace("/", "_")
    out_dir = company_file.parent

    detail_path = out_dir / f"{safe_query}_company_proxy_detail_{year}.csv"
    summary_path = out_dir / f"{safe_query}_company_proxy_summary_{year}.csv"

    detail_df.to_csv(detail_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    print()
    print("=" * 90)
    print("RESULT MEANING")
    print("=" * 90)
    print("This is NOT exact company count for Chemical Engineer.")
    print("This is related industry/business proxy.")
    print()
    print("Use this wording in your app:")
    print("  Company count: not available directly")
    print("  Related employer/business proxy: available by related industry")
    print("=" * 90)

    print()
    print("Rows found:", len(detail_df))
    print("Summary rows:", len(summary_df))

    print()
    print("Saved detail:")
    print(detail_path)

    print()
    print("Saved summary:")
    print(summary_path)

    print()
    print("Top summary rows:")
    display(summary_df.head(100).fillna(""))

    return detail_df, summary_df


# ============================================================
# RUN SEARCH
# ============================================================

detail_df, summary_df = search_company_proxy(
    company_file=COMPANY_FILE,
    query=QUERY,
    year=YEAR,
    chunksize=CHUNKSIZE
)

# “The industry related to this degree had openings, closings, job gains, job losses, and startup activity.”

# Related industry: Manufacturing
# In 2010, this industry had 4,363 job gains, 408 openings, 4,144 job losses, 524 closings, and 219 establishment births.

In [ ]:
from pathlib import Path
import pandas as pd

# Change this based on the degree search related industry
TARGET_YEAR = 2010
RELATED_INDUSTRY_KEYWORDS = ["manufacturing"]   # example for engineering-related broad industry

csv_path = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/3_Company/OneFile/business_startup_ALL_IN_ONE.csv"

keep_metrics = [
    "Gross Job Gains",
    "Openings",
    "Gross Job Losses",
    "Closings",
    "Establishment Births",
    "net_job_creation",
    "job_creation_rate_births",
]

usecols = [
    "source",
    "dataset",
    "source_table",
    "year",
    "state_name",
    "industry_code",
    "industry_name",
    "metric_name",
    "value",
]

pieces = []

print("Reading file in chunks, not loading full CSV...")

for chunk in pd.read_csv(
    csv_path,
    usecols=usecols,
    chunksize=200_000,
    low_memory=False
):
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    # keep only one year
    chunk = chunk[chunk["year"].eq(TARGET_YEAR)]

    # keep only important metrics
    chunk = chunk[chunk["metric_name"].isin(keep_metrics)]

    # keep related industry only
    industry_text = chunk["industry_name"].fillna("").str.lower()
    mask = False
    for word in RELATED_INDUSTRY_KEYWORDS:
        mask = mask | industry_text.str.contains(word.lower(), na=False)

    chunk = chunk[mask]

    if chunk.empty:
        continue

    grouped = (
        chunk.groupby(
            ["year", "state_name", "industry_code", "industry_name", "metric_name"],
            dropna=False
        )
        .agg(total_value=("value", "sum"))
        .reset_index()
    )

    pieces.append(grouped)

if not pieces:
    print("No matching short summary found. Try changing RELATED_INDUSTRY_KEYWORDS.")
else:
    summary = pd.concat(pieces, ignore_index=True)

    summary = (
        summary.groupby(
            ["year", "state_name", "industry_code", "industry_name", "metric_name"],
            dropna=False
        )
        .agg(total_value=("total_value", "sum"))
        .reset_index()
    )

    short_result = summary.pivot_table(
        index=["year", "state_name", "industry_code", "industry_name"],
        columns="metric_name",
        values="total_value",
        aggfunc="sum"
    ).reset_index()

    short_result.columns.name = None

    # Rename columns shorter
    short_result = short_result.rename(columns={
        "Gross Job Gains": "job_gains",
        "Openings": "openings",
        "Gross Job Losses": "job_losses",
        "Closings": "closings",
        "Establishment Births": "business_births",
        "net_job_creation": "net_job_creation",
        "job_creation_rate_births": "birth_job_creation_rate",
    })

    print("Short degree-related industry summary:")
    display(short_result.head(20))

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

csv_path = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/3_Company/OneFile/business_startup_ALL_IN_ONE.csv"

START_YEAR = 2010
END_YEAR = 2025

# Change this for the degree-related industry
RELATED_INDUSTRY_KEYWORDS = ["manufacturing"]

keep_metrics = [
    "Gross Job Gains",
    "Openings",
    "Gross Job Losses",
    "Closings",
    "Establishment Births",
]

usecols = [
    "year",
    "industry_name",
    "metric_name",
    "value",
]

pieces = []

print("Building chart data from 2010 to 2025...")

for chunk in pd.read_csv(
    csv_path,
    usecols=usecols,
    chunksize=200_000,
    low_memory=False
):
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    chunk = chunk[
        chunk["year"].between(START_YEAR, END_YEAR)
        & chunk["metric_name"].isin(keep_metrics)
    ]

    if chunk.empty:
        continue

    # FIX: convert industry_name to text safely
    industry_text = chunk["industry_name"].astype("string").fillna("").str.lower()

    mask = pd.Series(False, index=chunk.index)

    for word in RELATED_INDUSTRY_KEYWORDS:
        mask = mask | industry_text.str.contains(word.lower(), na=False)

    chunk = chunk[mask]

    if chunk.empty:
        continue

    grouped = (
        chunk.groupby(["year", "metric_name"], dropna=False)["value"]
        .sum()
        .reset_index()
    )

    pieces.append(grouped)

if not pieces:
    print("No data found. Try changing RELATED_INDUSTRY_KEYWORDS.")
else:
    chart_data = pd.concat(pieces, ignore_index=True)

    chart_data = (
        chart_data.groupby(["year", "metric_name"], dropna=False)["value"]
        .sum()
        .reset_index()
    )

    chart_wide = chart_data.pivot_table(
        index="year",
        columns="metric_name",
        values="value",
        aggfunc="sum"
    ).sort_index()

    print("Years found:", list(chart_wide.index.astype(int)))
    print("Metrics found:", list(chart_wide.columns))

    plt.figure(figsize=(12, 6))

    for col in chart_wide.columns:
        plt.plot(chart_wide.index, chart_wide[col], marker="o", label=col)

    plt.title("Degree-related industry trend: Manufacturing, 2010–2025")
    plt.xlabel("Year")
    plt.ylabel("Total value")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# key word search

In [ ]:
from pathlib import Path
import pandas as pd

csv_path = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/3_Company/OneFile/business_startup_ALL_IN_ONE.csv"

usecols = ["industry_code", "industry_name"]

all_industries = set()

print("Collecting all unique industry names...")

for chunk in pd.read_csv(
    csv_path,
    usecols=usecols,
    chunksize=200_000,
    dtype=str,
    low_memory=False
):
    chunk["industry_code"] = chunk["industry_code"].fillna("")
    chunk["industry_name"] = chunk["industry_name"].fillna("")

    temp = chunk[["industry_code", "industry_name"]].drop_duplicates()

    for row in temp.itertuples(index=False):
        if row.industry_name.strip() != "":
            all_industries.add((row.industry_code, row.industry_name))

industry_list = pd.DataFrame(
    sorted(all_industries),
    columns=["industry_code", "industry_name"]
)

print("Total unique industries:", len(industry_list))
display(industry_list.head(300))

# Save full list so you can open/search it
out_path = csv_path.parent / "industry_keyword_list.csv"
industry_list.to_csv(out_path, index=False)

print("Saved full industry keyword list here:")
print(out_path)

In [ ]:
from pathlib import Path
import pandas as pd

csv_path = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/3_Company/OneFile/business_startup_ALL_IN_ONE.csv"

usecols = ["industry_code", "industry_name"]

all_industries = set()

print("Reading all unique industry names...")

for chunk in pd.read_csv(
    csv_path,
    usecols=usecols,
    chunksize=200_000,
    dtype=str,
    low_memory=False
):
    chunk["industry_code"] = chunk["industry_code"].fillna("")
    chunk["industry_name"] = chunk["industry_name"].fillna("")

    temp = chunk[["industry_code", "industry_name"]].drop_duplicates()

    for row in temp.itertuples(index=False):
        if row.industry_name.strip() != "":
            all_industries.add((row.industry_code, row.industry_name))

industry_list = pd.DataFrame(
    sorted(all_industries),
    columns=["industry_code", "industry_name"]
)

print("Total unique industries:", len(industry_list))
print()
print(industry_list.to_string(index=False))

# Show all year

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
from IPython.display import display

# Search all years
RELATED_INDUSTRY_KEYWORDS = ["manufacturing"]   # change this for degree-related industry

csv_path = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/3_Company/OneFile/business_startup_ALL_IN_ONE.csv"

keep_metrics = [
    "Gross Job Gains",
    "Openings",
    "Gross Job Losses",
    "Closings",
    "Establishment Births",
    "net_job_creation",
    "job_creation_rate_births",
]

usecols = [
    "source",
    "dataset",
    "source_table",
    "year",
    "state_name",
    "industry_code",
    "industry_name",
    "metric_name",
    "value",
]

group_cols = [
    "year",
    "state_name",
    "industry_code",
    "industry_name",
    "metric_name",
]

# Memory optimized accumulator
summary_dict = defaultdict(float)

rows_checked = 0
rows_matched = 0

print("Reading file in chunks, searching ALL years...")

for i, chunk in enumerate(pd.read_csv(
    csv_path,
    usecols=usecols,
    chunksize=200_000,
    low_memory=False
), start=1):

    rows_checked += len(chunk)

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    # Keep valid year/value only
    chunk = chunk[
        chunk["year"].notna()
        & chunk["value"].notna()
    ]

    # Keep only important metrics
    chunk = chunk[chunk["metric_name"].isin(keep_metrics)]

    # Keep related industry only
    industry_text = chunk["industry_name"].fillna("").str.lower()

    mask = pd.Series(False, index=chunk.index)

    for word in RELATED_INDUSTRY_KEYWORDS:
        mask = mask | industry_text.str.contains(word.lower(), na=False, regex=False)

    chunk = chunk[mask]

    if chunk.empty:
        if i % 10 == 0:
            print(f"Chunk {i} | rows checked: {rows_checked:,} | matched: {rows_matched:,}")
        continue

    rows_matched += len(chunk)

    # Make year clean integer
    chunk["year"] = chunk["year"].astype(int)

    grouped = (
        chunk.groupby(group_cols, dropna=False)
        .agg(total_value=("value", "sum"))
        .reset_index()
    )

    # Add grouped results into dictionary instead of storing all chunks
    for row in grouped.itertuples(index=False):
        key = (
            row.year,
            row.state_name,
            row.industry_code,
            row.industry_name,
            row.metric_name,
        )
        summary_dict[key] += row.total_value

    if i % 10 == 0:
        print(f"Chunk {i} | rows checked: {rows_checked:,} | matched: {rows_matched:,}")

print()
print("Finished reading.")
print("Total rows checked:", f"{rows_checked:,}")
print("Total matched rows:", f"{rows_matched:,}")

if not summary_dict:
    print("No matching summary found. Try changing RELATED_INDUSTRY_KEYWORDS.")
else:
    summary = pd.DataFrame(
        [
            (*key, total_value)
            for key, total_value in summary_dict.items()
        ],
        columns=[
            "year",
            "state_name",
            "industry_code",
            "industry_name",
            "metric_name",
            "total_value",
        ]
    )

    short_result = summary.pivot_table(
        index=["year", "state_name", "industry_code", "industry_name"],
        columns="metric_name",
        values="total_value",
        aggfunc="sum"
    ).reset_index()

    short_result.columns.name = None

    short_result = short_result.rename(columns={
        "Gross Job Gains": "job_gains",
        "Openings": "openings",
        "Gross Job Losses": "job_losses",
        "Closings": "closings",
        "Establishment Births": "business_births",
        "net_job_creation": "net_job_creation",
        "job_creation_rate_births": "birth_job_creation_rate",
    })

    short_result = short_result.sort_values(
        ["year", "state_name", "industry_name"],
        na_position="last"
    )

    print()
    print("Short degree-related industry summary for ALL years:")
    display(short_result.head(20))

    print()
    print("Years found:")
    print(sorted(short_result["year"].dropna().unique()))

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
from IPython.display import display

# ============================================================
# SETTINGS
# ============================================================

# Search all years automatically
RELATED_INDUSTRY_KEYWORDS = ["manufacturing"]

csv_path = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/3_Company/OneFile/business_startup_ALL_IN_ONE.csv"

keep_metrics = [
    "Gross Job Gains",
    "Openings",
    "Gross Job Losses",
    "Closings",
    "Establishment Births",
    "net_job_creation",
    "job_creation_rate_births",
]

usecols = [
    "source",
    "dataset",
    "source_table",
    "year",
    "state_name",
    "industry_code",
    "industry_name",
    "metric_name",
    "value",
]

group_cols = [
    "year",
    "state_name",
    "industry_code",
    "industry_name",
    "metric_name",
]

# Display setting
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# ============================================================
# CHECK FILE
# ============================================================

print("Checking file:")
print(csv_path)
print()

if not csv_path.exists():
    raise FileNotFoundError(f"File not found: {csv_path}")

print("File exists.")
print("Size MB:", round(csv_path.stat().st_size / 1024 / 1024, 2))
print()

# ============================================================
# READ IN CHUNKS - MEMORY OPTIMIZED
# ============================================================

summary_dict = defaultdict(float)

rows_checked = 0
rows_after_metric_filter = 0
rows_matched = 0

print("Reading file in chunks, searching ALL years...")
print("Industry keywords:", RELATED_INDUSTRY_KEYWORDS)
print()

for i, chunk in enumerate(pd.read_csv(
    csv_path,
    usecols=usecols,
    chunksize=200_000,
    low_memory=False
), start=1):

    rows_checked += len(chunk)

    # Convert numbers safely
    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    # Keep valid year and value
    chunk = chunk[
        chunk["year"].notna()
        & chunk["value"].notna()
    ]

    # Keep only important metrics
    chunk = chunk[chunk["metric_name"].isin(keep_metrics)]
    rows_after_metric_filter += len(chunk)

    if chunk.empty:
        if i % 10 == 0:
            print(
                f"Chunk {i} | rows checked: {rows_checked:,} | "
                f"after metric filter: {rows_after_metric_filter:,} | "
                f"matched: {rows_matched:,}"
            )
        continue

    # Search industry keyword
    industry_text = chunk["industry_name"].fillna("").str.lower()

    mask = pd.Series(False, index=chunk.index)

    for word in RELATED_INDUSTRY_KEYWORDS:
        mask = mask | industry_text.str.contains(
            word.lower(),
            na=False,
            regex=False
        )

    chunk = chunk[mask]

    if chunk.empty:
        if i % 10 == 0:
            print(
                f"Chunk {i} | rows checked: {rows_checked:,} | "
                f"after metric filter: {rows_after_metric_filter:,} | "
                f"matched: {rows_matched:,}"
            )
        continue

    rows_matched += len(chunk)

    # Clean year
    chunk["year"] = chunk["year"].astype(int)

    # Group inside each chunk first
    grouped = (
        chunk.groupby(group_cols, dropna=False)
        .agg(total_value=("value", "sum"))
        .reset_index()
    )

    # Add into dictionary instead of storing all chunks
    for row in grouped.itertuples(index=False):
        key = (
            row.year,
            row.state_name,
            row.industry_code,
            row.industry_name,
            row.metric_name,
        )
        summary_dict[key] += row.total_value

    if i % 10 == 0:
        print(
            f"Chunk {i} | rows checked: {rows_checked:,} | "
            f"after metric filter: {rows_after_metric_filter:,} | "
            f"matched: {rows_matched:,}"
        )

print()
print("Finished reading.")
print("Total rows checked:", f"{rows_checked:,}")
print("Rows after metric filter:", f"{rows_after_metric_filter:,}")
print("Total matched rows:", f"{rows_matched:,}")
print()

# ============================================================
# BUILD RESULT
# ============================================================

if not summary_dict:
    print("No matching result found.")
    print("Try changing RELATED_INDUSTRY_KEYWORDS.")
else:
    summary = pd.DataFrame(
        [
            (*key, total_value)
            for key, total_value in summary_dict.items()
        ],
        columns=[
            "year",
            "state_name",
            "industry_code",
            "industry_name",
            "metric_name",
            "total_value",
        ]
    )

    short_result = summary.pivot_table(
        index=[
            "year",
            "state_name",
            "industry_code",
            "industry_name",
        ],
        columns="metric_name",
        values="total_value",
        aggfunc="sum"
    ).reset_index()

    short_result.columns.name = None

    # Rename columns shorter
    short_result = short_result.rename(columns={
        "Gross Job Gains": "job_gains",
        "Openings": "openings",
        "Gross Job Losses": "job_losses",
        "Closings": "closings",
        "Establishment Births": "business_births",
        "net_job_creation": "net_job_creation",
        "job_creation_rate_births": "birth_job_creation_rate",
    })

    short_result = short_result.sort_values(
        [
            "year",
            "state_name",
            "industry_code",
            "industry_name",
        ],
        na_position="last"
    ).reset_index(drop=True)

    # ========================================================
    # PRINT SUMMARY INFO
    # ========================================================

    print("Result rows:", len(short_result))
    print()

    print("Years found:")
    print(sorted(short_result["year"].dropna().unique()))
    print()

    print("Matching industries found:")
    industry_list = (
        short_result[["industry_code", "industry_name"]]
        .drop_duplicates()
        .sort_values(["industry_code", "industry_name"])
        .reset_index(drop=True)
    )
    display(industry_list)

    print("States found:")
    state_list = (
        short_result[["state_name"]]
        .drop_duplicates()
        .sort_values("state_name")
        .reset_index(drop=True)
    )
    display(state_list)

    # ========================================================
    # DISPLAY RESULT
    # ========================================================

    print("First 100 result rows:")
    display(short_result.head(100))

    print("Last 100 result rows:")
    display(short_result.tail(100))

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
from IPython.display import display

# ============================================================
# SETTINGS
# ============================================================

RELATED_INDUSTRY_KEYWORDS = ["manufacturing"]

csv_path = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/3_Company/OneFile/business_startup_ALL_IN_ONE.csv"

keep_metrics = [
    "Gross Job Gains",
    "Openings",
    "Gross Job Losses",
    "Closings",
    "Establishment Births",
    "net_job_creation",
    "job_creation_rate_births",
]

usecols = [
    "source",
    "dataset",
    "source_table",
    "year",
    "state_name",
    "industry_code",
    "industry_name",
    "metric_name",
    "value",
]

group_cols = [
    "year",
    "state_name",
    "industry_code",
    "industry_name",
    "metric_name",
]

# Show everything in Jupyter
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# ============================================================
# CHECK FILE
# ============================================================

print("Checking file:")
print(csv_path)
print()

if not csv_path.exists():
    raise FileNotFoundError(f"File not found: {csv_path}")

print("File exists.")
print("Size MB:", round(csv_path.stat().st_size / 1024 / 1024, 2))
print()

# ============================================================
# READ IN CHUNKS - MEMORY OPTIMIZED
# ============================================================

summary_dict = defaultdict(float)

rows_checked = 0
rows_after_metric_filter = 0
rows_matched = 0

print("Reading file in chunks, searching ALL years...")
print("Industry keywords:", RELATED_INDUSTRY_KEYWORDS)
print()

for i, chunk in enumerate(pd.read_csv(
    csv_path,
    usecols=usecols,
    chunksize=200_000,
    low_memory=False
), start=1):

    rows_checked += len(chunk)

    chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
    chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")

    chunk = chunk[
        chunk["year"].notna()
        & chunk["value"].notna()
    ]

    chunk = chunk[chunk["metric_name"].isin(keep_metrics)]
    rows_after_metric_filter += len(chunk)

    if chunk.empty:
        if i % 10 == 0:
            print(
                f"Chunk {i} | rows checked: {rows_checked:,} | "
                f"after metric filter: {rows_after_metric_filter:,} | "
                f"matched: {rows_matched:,}"
            )
        continue

    industry_text = chunk["industry_name"].fillna("").str.lower()

    mask = pd.Series(False, index=chunk.index)

    for word in RELATED_INDUSTRY_KEYWORDS:
        mask = mask | industry_text.str.contains(
            word.lower(),
            na=False,
            regex=False
        )

    chunk = chunk[mask]

    if chunk.empty:
        if i % 10 == 0:
            print(
                f"Chunk {i} | rows checked: {rows_checked:,} | "
                f"after metric filter: {rows_after_metric_filter:,} | "
                f"matched: {rows_matched:,}"
            )
        continue

    rows_matched += len(chunk)

    chunk["year"] = chunk["year"].astype(int)

    grouped = (
        chunk.groupby(group_cols, dropna=False)
        .agg(total_value=("value", "sum"))
        .reset_index()
    )

    for row in grouped.itertuples(index=False):
        key = (
            row.year,
            row.state_name,
            row.industry_code,
            row.industry_name,
            row.metric_name,
        )
        summary_dict[key] += row.total_value

    if i % 10 == 0:
        print(
            f"Chunk {i} | rows checked: {rows_checked:,} | "
            f"after metric filter: {rows_after_metric_filter:,} | "
            f"matched: {rows_matched:,}"
        )

print()
print("Finished reading.")
print("Total rows checked:", f"{rows_checked:,}")
print("Rows after metric filter:", f"{rows_after_metric_filter:,}")
print("Total matched rows:", f"{rows_matched:,}")
print()

# ============================================================
# BUILD FINAL RESULT
# ============================================================

if not summary_dict:
    print("No matching result found.")
    print("Try changing RELATED_INDUSTRY_KEYWORDS.")
else:
    summary = pd.DataFrame(
        [
            (*key, total_value)
            for key, total_value in summary_dict.items()
        ],
        columns=[
            "year",
            "state_name",
            "industry_code",
            "industry_name",
            "metric_name",
            "total_value",
        ]
    )

    short_result = summary.pivot_table(
        index=[
            "year",
            "state_name",
            "industry_code",
            "industry_name",
        ],
        columns="metric_name",
        values="total_value",
        aggfunc="sum"
    ).reset_index()

    short_result.columns.name = None

    short_result = short_result.rename(columns={
        "Gross Job Gains": "job_gains",
        "Openings": "openings",
        "Gross Job Losses": "job_losses",
        "Closings": "closings",
        "Establishment Births": "business_births",
        "net_job_creation": "net_job_creation",
        "job_creation_rate_births": "birth_job_creation_rate",
    })

    short_result = short_result.sort_values(
        [
            "year",
            "state_name",
            "industry_code",
            "industry_name",
        ],
        na_position="last"
    ).reset_index(drop=True)

    print("Result rows:", len(short_result))
    print()

    print("Years found:")
    print(sorted(short_result["year"].dropna().unique()))
    print()

    print("Matching industries found:")
    industry_list = (
        short_result[["industry_code", "industry_name"]]
        .drop_duplicates()
        .sort_values(["industry_code", "industry_name"])
        .reset_index(drop=True)
    )
    display(industry_list)

    print("States found:")
    state_list = (
        short_result[["state_name"]]
        .drop_duplicates()
        .sort_values("state_name")
        .reset_index(drop=True)
    )
    display(state_list)

    print("ALL result rows:")
    display(short_result)

# key work change fix

In [ ]:
industry_label_map = {
    "NAICS11": "Agriculture, Forestry, Fishing and Hunting",
    "NAICS21": "Mining, Quarrying, and Oil/Gas Extraction",
    "NAICS22": "Utilities",
    "NAICS23": "Construction",
    "NAICS42": "Wholesale Trade",
    "NAICS51": "Information",
    "NAICS52": "Finance and Insurance",
    "NAICS53": "Real Estate and Rental and Leasing",
    "NAICS54": "Professional, Scientific, and Technical Services",
    "NAICS55": "Management of Companies and Enterprises",
    "NAICS56": "Administrative/Support/Waste Management",
    "NAICS61": "Educational Services",
    "NAICS62": "Health Care and Social Assistance",
    "NAICS71": "Arts, Entertainment, and Recreation",
    "NAICS72": "Accommodation and Food Services",
    "NAICS81": "Other Services except Public Administration",
    "NAICSMNF": "Manufacturing",
    "NAICSRET": "Retail Trade",
    "NAICSTW": "Transportation and Warehousing",
    "NONAICS": "Not classified",
    "TOTAL": "Total / all industries",
}

df["industry_name_clean"] = df["industry_code"].map(industry_label_map).fillna(df["industry_name"])

In [ ]:
from pathlib import Path
import pandas as pd
import time

# ============================================================
# SEARCH INDUSTRY KEYWORDS IN BIG business_startup_ALL_IN_ONE.csv
# Memory optimized: chunk read, selected columns only, no saving
# ============================================================

FILE = Path.home() / "Downloads" / "Internship_SCIPE CI-SIP" / "MainProject" / "3_Company" / "OneFile" / "business_startup_ALL_IN_ONE.csv"

# Change this keyword
SEARCH_KEYWORD = "manufacturing"

CHUNKSIZE = 300_000

industry_label_map = {
    "NAICS11": "Agriculture, Forestry, Fishing and Hunting",
    "NAICS21": "Mining, Quarrying, and Oil/Gas Extraction",
    "NAICS22": "Utilities",
    "NAICS23": "Construction",
    "NAICS42": "Wholesale Trade",
    "NAICS51": "Information",
    "NAICS52": "Finance and Insurance",
    "NAICS53": "Real Estate and Rental and Leasing",
    "NAICS54": "Professional, Scientific, and Technical Services",
    "NAICS55": "Management of Companies and Enterprises",
    "NAICS56": "Administrative and Support and Waste Management",
    "NAICS61": "Educational Services",
    "NAICS62": "Health Care and Social Assistance",
    "NAICS71": "Arts, Entertainment, and Recreation",
    "NAICS72": "Accommodation and Food Services",
    "NAICS81": "Other Services except Public Administration",
    "NAICSMNF": "Manufacturing",
    "NAICSRET": "Retail Trade",
    "NAICSTW": "Transportation and Warehousing",
    "NONAICS": "Not classified",
    "TOTAL": "Total / all industries",
}

usecols = [
    "source",
    "dataset",
    "year",
    "state_name",
    "industry_code",
    "industry_name",
    "metric_name",
    "value",
]

print("Checking file:")
print(FILE)

if not FILE.exists():
    raise FileNotFoundError(f"File not found:\n{FILE}")

print("File exists.")
print("Searching keyword:", SEARCH_KEYWORD)
print()

start = time.time()
matches = []
unique_industries = {}
rows_checked = 0

for i, chunk in enumerate(pd.read_csv(FILE, usecols=usecols, chunksize=CHUNKSIZE, low_memory=False), start=1):
    rows_checked += len(chunk)

    # Clean columns
    chunk["industry_code"] = chunk["industry_code"].astype(str).str.strip()
    chunk["industry_name"] = chunk["industry_name"].astype(str).str.strip()

    # Fix NAICS codes where industry_name only says NAICS11, NAICSMNF, etc.
    chunk["industry_name_clean"] = chunk["industry_code"].map(industry_label_map).fillna(chunk["industry_name"])

    # Search both code and clean name
    search_text = (
        chunk["industry_code"].fillna("").str.lower()
        + " "
        + chunk["industry_name_clean"].fillna("").str.lower()
    )

    mask = search_text.str.contains(SEARCH_KEYWORD.lower(), na=False)

    found = chunk.loc[mask].copy()

    if not found.empty:
        matches.append(found)

        temp_unique = found[["industry_code", "industry_name_clean"]].drop_duplicates()

        for _, row in temp_unique.iterrows():
            unique_industries[row["industry_code"]] = row["industry_name_clean"]

    if i % 5 == 0:
        elapsed = round(time.time() - start, 1)
        print(f"Chunk {i} | rows checked {rows_checked:,} | matches so far {sum(len(x) for x in matches):,} | elapsed {elapsed}s")

print()
print("DONE")
print("Total rows checked:", f"{rows_checked:,}")
print("Total matched rows:", f"{sum(len(x) for x in matches):,}")
print()

# ============================================================
# SHOW UNIQUE INDUSTRIES FOUND
# ============================================================

unique_df = pd.DataFrame(
    sorted(unique_industries.items()),
    columns=["industry_code", "industry_name_clean"]
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

print("Unique matching industries:")
display(unique_df)

# ============================================================
# SHOW MATCHING DATA
# ============================================================

if matches:
    result = pd.concat(matches, ignore_index=True)

    result["value"] = pd.to_numeric(result["value"], errors="coerce")

    result = result[
        [
            "source",
            "dataset",
            "year",
            "state_name",
            "industry_code",
            "industry_name_clean",
            "metric_name",
            "value",
        ]
    ].sort_values(
        ["year", "state_name", "industry_code", "metric_name"],
        na_position="last"
    )

    print("Matching data rows:")
    display(result)
else:
    print("No matches found.")